<div dir="rtl" align="right">

# نوت‌بوک ۱۴ — ساخت Dense Index نسخه‌دار

## هدف

این نوت‌بوک بردارهای Embedding منتخب و فریز‌شده در NB13 را بدون محاسبه مجدد Embedding، داخل یک Dense Index نسخه‌دار در Qdrant Local ذخیره می‌کند.

هر Point باید به Chunk، Golden Dataset، مدل Embedding، Revision و Release مشخص قابل ردیابی باشد.

## جایگاه در Pipeline

Golden Dataset فریز‌شده ← Embedding Benchmark فریز‌شده ← **Dense Index** ← Sparse Index ← Hybrid Retrieval ← Reranking

## ورودی‌های اصلی

- قرارداد رسمی تحویل NB13 به NB14
- Embedding Release فریز‌شده
- Chunk release منتخب
- Cache کامل Embeddingهای E5
- تنظیمات نسخه‌دار Qdrant Collection

## خروجی‌های مورد انتظار

- Qdrant Collection نسخه‌دار
- Payload schema قابل ردیابی
- Batch upsert قابل Resume
- Point-count و ID-parity evidence
- Payload و Vector spot-check
- Snapshot و Restore evidence
- Dense Index manifest
- Acceptance report و Gate نهایی NB14

## محدوده

- ساخت و اعتبارسنجی Dense Index
- حفظ کامل lineage
- قابلیت اجرای مجدد و Resume
- استفاده از Embeddingهای Cacheشده

## خارج از محدوده

- محاسبه مجدد Embedding
- API یا مدل پولی
- Sparse/BM25
- Hybrid Retrieval
- Reranking
- تغییر Golden Dataset، Chunk release یا Embedding configuration

## معیار تکمیل

نوت‌بوک فقط زمانی کامل است که Execution Health و تمام Deliverableهای Roadmap پاس شوند، تعداد Pointها با Chunkها برابر باشد، lineage معتبر باشد، Snapshot/Restore آزمایش شود و Index manifest فریز‌شده تولید گردد.

</div>

<div dir="rtl" align="right">

## مرحله ۱ — Bootstrap و Preflight ورودی‌های فریز‌شده

NB13 با وضعیت `FROZEN_FOR_NB14` بسته شده و قرارداد تحویل آن `READY_FOR_NB14` است.

در این مرحله ابتدا بدون ایجاد یا تغییر هیچ Artifactی، موارد زیر از روی Disk واقعی بررسی می‌شوند:

- وجود و Hash فایل Embedding Release
- وجود و Hash قرارداد Handoff
- تطابق Golden Dataset SHA
- تطابق مدل، Revision، Dimension، Prefixها و Normalization
- وجود Chunk artifact منتخب
- تطابق تعداد و SHA فایل Chunk
- وجود مسیر Cache منتخب
- تعداد فایل‌های NPZ
- نصب بودن Qdrant Client

این مرحله هیچ Collectionی نمی‌سازد، هیچ برداری را Upsert نمی‌کند و هیچ Embedding جدیدی تولید نمی‌کند.

موفقیت این مرحله زمانی اعلام می‌شود که تمام ورودی‌های ضروری موجود و با قرارداد فریز‌شده سازگار باشند.

</div>

In [1]:
# Code Cell 01 — Centralized imports / read-only bootstrap

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path
from typing import Any

import numpy as np
from qdrant_client import QdrantClient

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("qdrant-client:", importlib.metadata.version("qdrant-client"))
print("Bootstrap status: PASS")

Python: 3.11.3
Platform: macOS-12.7.6-x86_64-i386-64bit
NumPy: 1.24.3
qdrant-client: 1.18.0
Bootstrap status: PASS


In [2]:
# Code Cell 02 — Read-only NB14 canonical handoff preflight

EXPECTED_HANDOFF_SHA256 = (
    "77eb9f34e36f174cf748fd99caa1d507b5dc8e4abcb51f62b452c35d46688efc"
)
EXPECTED_RELEASE_SHA256 = (
    "8c48acc7c266dd153bce038ff6ec679b716e88679ccd865e15c333e4fbd7f429"
)


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    assert isinstance(value, dict), f"Expected JSON object: {path}"
    return value


def find_repo_root(start: Path) -> Path:
    required = Path("artifacts/phase2/embedding/embedding_release_v1.json")
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / required).is_file():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found from the current working directory."
    )


repo_root = find_repo_root(Path.cwd())

release_path = (
    repo_root / "artifacts/phase2/embedding/embedding_release_v1.json"
)
handoff_path = (
    repo_root / "artifacts/phase2/embedding/nb14_dense_index_handoff_v1.json"
)

assert release_path.is_file(), f"Missing release: {release_path}"
assert handoff_path.is_file(), f"Missing handoff: {handoff_path}"

release = load_json(release_path)
handoff = load_json(handoff_path)

release_sha = sha256_file(release_path)
handoff_sha = sha256_file(handoff_path)

assert release_sha == EXPECTED_RELEASE_SHA256, (
    f"Embedding release SHA mismatch: {release_sha}"
)
assert handoff_sha == EXPECTED_HANDOFF_SHA256, (
    f"NB14 handoff SHA mismatch: {handoff_sha}"
)

assert release["status"] == "FROZEN_FOR_NB14"
assert handoff["status"] == "READY_FOR_NB14"
assert handoff["embedding_release"]["sha256"] == release_sha
assert handoff["lineage"]["embedding_release_sha256"] == release_sha

release_embedding = release["embedding"]
handoff_embedding = handoff["embedding_config"]

embedding_contract_fields = (
    "model_id",
    "revision",
    "dimension",
    "query_prefix",
    "passage_prefix",
    "normalization",
    "snapshot_content_identity_sha256",
)

for field in embedding_contract_fields:
    assert release_embedding[field] == handoff_embedding[field], (
        f"Embedding contract mismatch for {field!r}"
    )

assert release_embedding["model_id"] == "intfloat/multilingual-e5-small"
assert release_embedding["revision"] == (
    "614241f622f53c4eeff9890bdc4f31cfecc418b3"
)
assert release_embedding["dimension"] == 384
assert release_embedding["query_prefix"] == "query:"
assert release_embedding["passage_prefix"] == "passage:"

assert (
    release["golden_dataset"]["sha256"]
    == handoff["golden_dataset_sha256"]
)
assert release["chunking"] == handoff["chunk_input"]

chunk_contract = handoff["chunk_input"]
chunk_path = repo_root / chunk_contract["path"]

assert chunk_path.is_file(), f"Missing selected chunk artifact: {chunk_path}"

actual_chunk_sha = sha256_file(chunk_path)
assert actual_chunk_sha == chunk_contract["sha256"], (
    f"Chunk SHA mismatch: {actual_chunk_sha}"
)

with chunk_path.open("r", encoding="utf-8") as handle:
    actual_chunk_count = sum(1 for line in handle if line.strip())

assert actual_chunk_count == chunk_contract["record_count"], (
    f"Chunk-count mismatch: {actual_chunk_count}"
)

cache_contract = handoff["vector_cache"]
cache_root = repo_root / cache_contract["root"]

assert cache_root.is_dir(), f"Missing embedding cache root: {cache_root}"
assert cache_contract["recompute_required"] is False

npz_files = sorted(cache_root.rglob("*.npz"))
json_metadata_files = sorted(cache_root.rglob("*.json"))

assert len(npz_files) == release["cache"]["npz_file_count"], (
    f"Expected {release['cache']['npz_file_count']} NPZ files, "
    f"found {len(npz_files)}"
)

preflight_summary = {
    "repo_root": str(repo_root),
    "release": {
        "path": str(release_path.relative_to(repo_root)),
        "sha256": release_sha,
        "status": release["status"],
    },
    "handoff": {
        "path": str(handoff_path.relative_to(repo_root)),
        "sha256": handoff_sha,
        "status": handoff["status"],
    },
    "golden_dataset_sha256": handoff["golden_dataset_sha256"],
    "embedding": {
        field: handoff_embedding[field]
        for field in embedding_contract_fields
    },
    "chunk": {
        "path": chunk_contract["path"],
        "strategy": chunk_contract["strategy"],
        "count": actual_chunk_count,
        "sha256": actual_chunk_sha,
    },
    "cache": {
        "root": cache_contract["root"],
        "npz_file_count": len(npz_files),
        "json_metadata_file_count": len(json_metadata_files),
        "expected_inventory_sha256": cache_contract["inventory_sha256"],
        "recompute_required": cache_contract["recompute_required"],
    },
    "paid_api_required": handoff["constraints"]["paid_api_required"],
    "preflight_status": "PASS",
}

print(json.dumps(preflight_summary, ensure_ascii=False, indent=2))

{
  "repo_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "release": {
    "path": "artifacts/phase2/embedding/embedding_release_v1.json",
    "sha256": "8c48acc7c266dd153bce038ff6ec679b716e88679ccd865e15c333e4fbd7f429",
    "status": "FROZEN_FOR_NB14"
  },
  "handoff": {
    "path": "artifacts/phase2/embedding/nb14_dense_index_handoff_v1.json",
    "sha256": "77eb9f34e36f174cf748fd99caa1d507b5dc8e4abcb51f62b452c35d46688efc",
    "status": "READY_FOR_NB14"
  },
  "golden_dataset_sha256": "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2",
  "embedding": {
    "model_id": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "dimension": 384,
    "query_prefix": "query:",
    "passage_prefix": "passage:",
    "normalization": "normalized SentenceTransformer encoding with L2-normalized retrieval vectors",
    "snapshot_content_identity_sha256": "0a487b28bf6bee275240c12ddc31ac3639ed367138e238

<div dir="rtl" align="right">

## مرحله ۲ — بررسی Schema و سازگاری Chunk با Cache

Preflight ورودی‌های فریز‌شده با موفقیت پاس شد.

پیش از طراحی Collection و Payload باید ساختار واقعی رکوردهای Chunk و فایل‌های NPZ بررسی شود تا مشخص گردد:

- شناسه پایدار هر Chunk در کدام Field قرار دارد؛
- متن قابل Index در کدام Field قرار دارد؛
- Payloadهای lineage چه Fieldهایی دارند؛
- آرایه Embedding داخل NPZ با چه نام و شکلی ذخیره شده است؛
- چه Metadata یا Text identifierهایی برای اتصال بردار به Chunk موجود است؛
- تعداد، Dimension و نوع داده بردارهای قابل استفاده چیست.

این مرحله فقط Discovery خواندنی است و هیچ Index، Collection یا Artifactی ایجاد نمی‌کند.

</div>

In [3]:
# Code Cell 03 — Read-only chunk/cache schema discovery

from collections import Counter, defaultdict

# ---------- Chunk schema discovery ----------

chunk_records: list[dict[str, Any]] = []

with chunk_path.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue

        record = json.loads(line)

        assert isinstance(record, dict), (
            f"Chunk line {line_number} is not a JSON object."
        )

        chunk_records.append(record)

assert len(chunk_records) == chunk_contract["record_count"]

field_presence = Counter()
field_types: dict[str, Counter] = defaultdict(Counter)

for record in chunk_records:
    for key, value in record.items():
        field_presence[key] += 1
        field_types[key][type(value).__name__] += 1

candidate_id_fields = (
    "chunk_id",
    "id",
    "point_id",
    "record_id",
)

candidate_text_fields = (
    "text",
    "chunk_text",
    "content",
    "normalized_text",
)

id_field_report = {}
for field in candidate_id_fields:
    values = [
        record.get(field)
        for record in chunk_records
        if record.get(field) not in (None, "")
    ]
    if values:
        id_field_report[field] = {
            "present": len(values),
            "unique": len(set(map(str, values))),
            "sample": str(values[0])[:160],
        }

text_field_report = {}
for field in candidate_text_fields:
    values = [
        record.get(field)
        for record in chunk_records
        if isinstance(record.get(field), str) and record.get(field).strip()
    ]
    if values:
        lengths = [len(value) for value in values]
        text_field_report[field] = {
            "present": len(values),
            "min_chars": min(lengths),
            "max_chars": max(lengths),
            "mean_chars": round(sum(lengths) / len(lengths), 2),
            "sample": values[0][:200],
        }

# ---------- NPZ schema discovery ----------

npz_schema_counter = Counter()
npz_file_reports = []
total_vector_rows_by_key = Counter()
npz_read_errors = []

for npz_path in npz_files:
    relative_path = str(npz_path.relative_to(cache_root))

    try:
        with np.load(npz_path, allow_pickle=False) as archive:
            file_schema = []
            file_report = {
                "path": relative_path,
                "arrays": {},
            }

            for key in archive.files:
                try:
                    array = archive[key]
                    descriptor = {
                        "shape": list(array.shape),
                        "dtype": str(array.dtype),
                    }

                    if array.ndim == 2 and array.shape[1] == release_embedding["dimension"]:
                        descriptor["is_embedding_matrix"] = True
                        total_vector_rows_by_key[key] += int(array.shape[0])
                    else:
                        descriptor["is_embedding_matrix"] = False

                    if array.ndim == 1 and array.size:
                        first_value = array[0]
                        if isinstance(first_value, (str, np.str_)):
                            descriptor["sample"] = str(first_value)[:160]
                        elif np.isscalar(first_value):
                            descriptor["sample"] = str(first_value)[:160]

                    file_report["arrays"][key] = descriptor
                    file_schema.append(
                        (key, tuple(array.shape), str(array.dtype))
                    )

                except ValueError as exc:
                    file_report["arrays"][key] = {
                        "read_error": str(exc),
                        "possible_object_array": True,
                    }
                    file_schema.append((key, "UNREADABLE_WITHOUT_PICKLE"))

            npz_schema_counter[tuple(file_schema)] += 1
            npz_file_reports.append(file_report)

    except Exception as exc:
        npz_read_errors.append(
            {
                "path": relative_path,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
        )

assert not npz_read_errors, (
    f"Unreadable NPZ files detected: {npz_read_errors[:3]}"
)

representative_indexes = sorted(
    set(
        index
        for index in (0, 1, 2, len(npz_file_reports) - 1)
        if 0 <= index < len(npz_file_reports)
    )
)

schema_discovery = {
    "chunk_records": len(chunk_records),
    "chunk_fields": {
        field: {
            "present": field_presence[field],
            "types": dict(field_types[field]),
        }
        for field in sorted(field_presence)
    },
    "candidate_id_fields": id_field_report,
    "candidate_text_fields": text_field_report,
    "cache": {
        "npz_files": len(npz_files),
        "distinct_npz_schemas": len(npz_schema_counter),
        "embedding_rows_by_array_key": dict(total_vector_rows_by_key),
        "representative_files": [
            npz_file_reports[index]
            for index in representative_indexes
        ],
    },
    "discovery_status": "PASS",
}

print(json.dumps(schema_discovery, ensure_ascii=False, indent=2))

{
  "chunk_records": 3488,
  "chunk_fields": {
    "attribution_context": {
      "present": 3488,
      "types": {
        "dict": 3488
      }
    },
    "boundary_metadata": {
      "present": 3488,
      "types": {
        "dict": 3488
      }
    },
    "char_count": {
      "present": 3488,
      "types": {
        "int": 3488
      }
    },
    "chunk_contract_version": {
      "present": 3488,
      "types": {
        "str": 3488
      }
    },
    "chunk_count": {
      "present": 3488,
      "types": {
        "int": 3488
      }
    },
    "chunk_id": {
      "present": 3488,
      "types": {
        "str": 3488
      }
    },
    "chunk_index": {
      "present": 3488,
      "types": {
        "int": 3488
      }
    },
    "content": {
      "present": 3488,
      "types": {
        "str": 3488
      }
    },
    "content_origin": {
      "present": 3488,
      "types": {
        "list": 3488
      }
    },
    "lineage": {
      "present": 3488,
      "types": {
        "

<div dir="rtl" align="right">

## مرحله ۳ — اثبات اتصال قطعی Chunkها به بردارهای Cache

وجود ۹۰ فایل NPZ به‌تنهایی برای ساخت Index کافی نیست. باید ثابت شود هر Chunk منتخب دقیقاً به یک بردار Passage در Cache متصل می‌شود.

در این مرحله:

- تمام `input_keys` و بردارها به‌صورت خواندنی بررسی می‌شوند؛
- تعداد Query و Passage keyها مشخص می‌شود؛
- الگوریتم ساخت Passage key از متن واقعی Chunk آزموده می‌شود؛
- پوشش کامل ۳۴۸۸ Chunk بررسی می‌شود؛
- نبود بردار مفقود، Dimension اشتباه، مقدار غیرمتناهی و تعارض میان کلیدهای تکراری اثبات می‌شود؛
- Norm بردارها برای سازگاری با قرارداد L2 بررسی می‌شود.

تا زمانی که اتصال کامل Chunk → Cache Vector اثبات نشود، ساخت Qdrant Collection آغاز نخواهد شد.

</div>

In [4]:
# Code Cell 04 — Final deterministic chunk-to-cache mapping validation

def canonical_exact_text(text: Any) -> str:
    return " ".join(str(text).split())


def text_sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


cache_vectors: dict[str, np.ndarray] = {}
prefix_counts = Counter()
embedding_dtypes = Counter()
embedding_dimensions = Counter()
duplicate_keys = []
conflicting_keys = []

for npz_path in npz_files:
    with np.load(npz_path, allow_pickle=False) as archive:
        input_keys = archive["input_keys"]
        embeddings = archive["embeddings"]

        assert input_keys.ndim == 1
        assert embeddings.ndim == 2
        assert len(input_keys) == embeddings.shape[0]

        embedding_dtypes[str(embeddings.dtype)] += len(embeddings)
        embedding_dimensions[int(embeddings.shape[1])] += len(embeddings)

        for raw_key, raw_vector in zip(input_keys, embeddings):
            key = str(raw_key)
            vector = np.asarray(raw_vector, dtype=np.float32)

            prefix_counts[key.split(":", 1)[0]] += 1

            if key in cache_vectors:
                duplicate_keys.append(key)

                if not np.array_equal(cache_vectors[key], vector):
                    conflicting_keys.append(key)
            else:
                cache_vectors[key] = vector.copy()

assert not duplicate_keys, (
    f"Duplicate cache keys detected: {duplicate_keys[:5]}"
)
assert not conflicting_keys, (
    f"Conflicting cached vectors detected: {conflicting_keys[:5]}"
)
assert embedding_dimensions == Counter({384: 11433}), (
    f"Unexpected dimensions: {embedding_dimensions}"
)
assert embedding_dtypes == Counter({"float32": 11433}), (
    f"Unexpected dtypes: {embedding_dtypes}"
)

canonical_chunk_texts = [
    canonical_exact_text(record["content"])
    for record in chunk_records
]

chunk_cache_keys = [
    f"passage:{text_sha256(text)}"
    for text in canonical_chunk_texts
]

missing_chunk_indexes = [
    index
    for index, key in enumerate(chunk_cache_keys)
    if key not in cache_vectors
]

assert not missing_chunk_indexes, (
    "Selected chunks missing from the frozen E5 cache. "
    f"Examples: {missing_chunk_indexes[:10]}"
)

selected_vectors = np.stack(
    [cache_vectors[key] for key in chunk_cache_keys],
    axis=0,
).astype(np.float32, copy=False)

assert selected_vectors.shape == (
    chunk_contract["record_count"],
    release_embedding["dimension"],
)
assert np.isfinite(selected_vectors).all(), (
    "Selected vectors contain NaN or infinity."
)

vector_norms = np.linalg.norm(selected_vectors, axis=1)
norm_deviation = np.abs(vector_norms - 1.0)

assert np.all(vector_norms > 0)
assert float(norm_deviation.max()) <= 1e-4, (
    "Selected vectors violate the frozen L2-normalization contract. "
    f"Maximum deviation: {float(norm_deviation.max())}"
)

chunk_ids = [record["chunk_id"] for record in chunk_records]

assert len(chunk_ids) == len(set(chunk_ids)) == 3488

mapping_summary = {
    "cache_rows": sum(prefix_counts.values()),
    "unique_cache_keys": len(cache_vectors),
    "cache_prefix_counts": dict(sorted(prefix_counts.items())),
    "embedding_dtypes": dict(embedding_dtypes),
    "embedding_dimensions": dict(embedding_dimensions),
    "canonicalization": 'return " ".join(str(text).split())',
    "cache_key_contract": (
        "passage: + sha256(canonical_exact_text(content).encode('utf-8'))"
    ),
    "selected_chunks": len(chunk_records),
    "unique_chunk_ids": len(set(chunk_ids)),
    "unique_canonical_passages": len(set(canonical_chunk_texts)),
    "unique_selected_cache_keys": len(set(chunk_cache_keys)),
    "missing_chunk_vectors": len(missing_chunk_indexes),
    "selected_vector_shape": list(selected_vectors.shape),
    "vector_norm": {
        "min": float(vector_norms.min()),
        "max": float(vector_norms.max()),
        "mean": float(vector_norms.mean()),
        "max_deviation_from_one": float(norm_deviation.max()),
    },
    "chunk_to_cache_mapping_status": "PASS",
}

print(json.dumps(mapping_summary, ensure_ascii=False, indent=2))

{
  "cache_rows": 11433,
  "unique_cache_keys": 11433,
  "cache_prefix_counts": {
    "passage": 7698,
    "query": 3735
  },
  "embedding_dtypes": {
    "float32": 11433
  },
  "embedding_dimensions": {
    "384": 11433
  },
  "canonicalization": "return \" \".join(str(text).split())",
  "cache_key_contract": "passage: + sha256(canonical_exact_text(content).encode('utf-8'))",
  "selected_chunks": 3488,
  "unique_chunk_ids": 3488,
  "unique_canonical_passages": 3476,
  "unique_selected_cache_keys": 3476,
  "missing_chunk_vectors": 0,
  "selected_vector_shape": [
    3488,
    384
  ],
  "vector_norm": {
    "min": 0.9999998807907104,
    "max": 1.0000001192092896,
    "mean": 1.0,
    "max_deviation_from_one": 1.1920928955078125e-07
  },
  "chunk_to_cache_mapping_status": "PASS"
}


<div dir="rtl" align="right">

## مرحله ۴ — تعریف قرارداد نسخه‌دار Dense Index

ورودی Dense Index اکنون به‌طور کامل اثبات شده است:

- ۳۴۸۸ Chunk با شناسه یکتا؛
- ۳۴۷۶ متن canonical یکتا؛
- پوشش کامل Cache؛
- بردارهای `float32` با Dimension برابر ۳۸۴؛
- L2 normalization معتبر؛
- بدون بردار مفقود یا متعارض.

در این مرحله قرارداد ساخت Dense Index تعریف می‌شود، اما هنوز هیچ Collection یا فایلی ساخته نمی‌شود.

قرارداد شامل موارد زیر است:

- Index release ID؛
- مسیر نسخه‌دار Qdrant Local؛
- نام Collection نسخه‌دار؛
- فاصله Cosine؛
- Dimension برابر ۳۸۴؛
- UUID قطعی هر Point براساس `chunk_id`؛
- Payloadهای لازم برای lineage؛
- سیاست Batch و Resume؛
- اتصال مستقیم به Release و Handoff فریز‌شده.

همچنین مسیرهای موجود بررسی می‌شوند تا Collection یا Index قبلی ناخواسته overwrite نشود.

</div>

In [5]:
# Code Cell 05 — Define and preflight the versioned dense-index contract

import uuid

DENSE_INDEX_RELEASE_ID = "dense_index_v1"
COLLECTION_NAME = "rag_finance_phase2_dense_v1"

dense_index_root = (
    repo_root
    / "data/phase2/index"
    / DENSE_INDEX_RELEASE_ID
)

qdrant_storage_path = dense_index_root / "qdrant"
checkpoint_path = dense_index_root / "upsert_checkpoint.json"
index_manifest_path = dense_index_root / "dense_index_manifest.json"
snapshot_evidence_path = dense_index_root / "snapshot_restore_evidence.json"

POINT_UUID_NAMESPACE = uuid.UUID(
    "4dd47989-97bb-5e13-bc0f-fae8fa884248"
)

payload_fields = [
    "chunk_id",
    "chunk_index",
    "source_id",
    "source_type",
    "strategy_id",
    "strategy_version",
    "chunk_contract_version",
    "role",
    "parent_chunk_id",
    "char_count",
    "token_count",
    "content",
    "content_origin",
    "member_document_ids",
    "member_locators",
    "locator_start",
    "locator_end",
    "attribution_context",
    "rights",
    "lineage",
    "golden_dataset_sha256",
    "chunk_release_sha256",
    "embedding_model_id",
    "embedding_revision",
    "embedding_dimension",
    "embedding_release_sha256",
    "dense_index_release_id",
]

point_ids = [
    str(uuid.uuid5(POINT_UUID_NAMESPACE, chunk_id))
    for chunk_id in chunk_ids
]

assert len(point_ids) == len(set(point_ids)) == len(chunk_records)

existing_index_state = {
    "dense_index_root_exists": dense_index_root.exists(),
    "qdrant_storage_exists": qdrant_storage_path.exists(),
    "checkpoint_exists": checkpoint_path.is_file(),
    "manifest_exists": index_manifest_path.is_file(),
    "snapshot_evidence_exists": snapshot_evidence_path.is_file(),
}

existing_qdrant_entries = []

if qdrant_storage_path.is_dir():
    existing_qdrant_entries = sorted(
        str(path.relative_to(qdrant_storage_path))
        for path in qdrant_storage_path.rglob("*")
        if path.is_file()
    )

dense_index_contract = {
    "schema_version": "1.0",
    "index_release_id": DENSE_INDEX_RELEASE_ID,
    "backend": "qdrant_local",
    "qdrant_client_version": importlib.metadata.version("qdrant-client"),
    "storage_path": str(qdrant_storage_path.relative_to(repo_root)),
    "collection_name": COLLECTION_NAME,
    "vector": {
        "dimension": release_embedding["dimension"],
        "distance": "COSINE",
        "dtype": "float32",
        "normalization": release_embedding["normalization"],
    },
    "point_identity": {
        "source_field": "chunk_id",
        "algorithm": "UUIDv5",
        "namespace": str(POINT_UUID_NAMESPACE),
        "unique_point_ids": len(set(point_ids)),
    },
    "inputs": {
        "embedding_release_path": str(
            release_path.relative_to(repo_root)
        ),
        "embedding_release_sha256": release_sha,
        "handoff_path": str(handoff_path.relative_to(repo_root)),
        "handoff_sha256": handoff_sha,
        "golden_dataset_sha256": handoff["golden_dataset_sha256"],
        "chunk_path": chunk_contract["path"],
        "chunk_sha256": actual_chunk_sha,
        "chunk_count": len(chunk_records),
        "cache_root": cache_contract["root"],
        "cache_inventory_sha256": cache_contract["inventory_sha256"],
    },
    "payload_fields": payload_fields,
    "upsert_policy": {
        "batch_size": 128,
        "resume_checkpoint": str(checkpoint_path.relative_to(repo_root)),
        "idempotent_point_ids": True,
        "overwrite_existing_collection": False,
    },
    "planned_artifacts": {
        "manifest": str(index_manifest_path.relative_to(repo_root)),
        "snapshot_restore_evidence": str(
            snapshot_evidence_path.relative_to(repo_root)
        ),
    },
    "existing_state": existing_index_state,
    "existing_qdrant_file_count": len(existing_qdrant_entries),
    "existing_qdrant_file_examples": existing_qdrant_entries[:20],
    "contract_status": "PASS_READ_ONLY",
}

print(json.dumps(dense_index_contract, ensure_ascii=False, indent=2))

{
  "schema_version": "1.0",
  "index_release_id": "dense_index_v1",
  "backend": "qdrant_local",
  "qdrant_client_version": "1.18.0",
  "storage_path": "data/phase2/index/dense_index_v1/qdrant",
  "collection_name": "rag_finance_phase2_dense_v1",
  "vector": {
    "dimension": 384,
    "distance": "COSINE",
    "dtype": "float32",
    "normalization": "normalized SentenceTransformer encoding with L2-normalized retrieval vectors"
  },
  "point_identity": {
    "source_field": "chunk_id",
    "algorithm": "UUIDv5",
    "namespace": "4dd47989-97bb-5e13-bc0f-fae8fa884248",
    "unique_point_ids": 3488
  },
  "inputs": {
    "embedding_release_path": "artifacts/phase2/embedding/embedding_release_v1.json",
    "embedding_release_sha256": "8c48acc7c266dd153bce038ff6ec679b716e88679ccd865e15c333e4fbd7f429",
    "handoff_path": "artifacts/phase2/embedding/nb14_dense_index_handoff_v1.json",
    "handoff_sha256": "77eb9f34e36f174cf748fd99caa1d507b5dc8e4abcb51f62b452c35d46688efc",
    "golden_data

<div dir="rtl" align="right">

## مرحله ۵ — ساخت Idempotent Collection و Payload Schema

قرارداد Dense Index تثبیت شد و هیچ Index قبلی در مسیر مقصد وجود ندارد.

در این مرحله:

- مسیر نسخه‌دار Qdrant Local ایجاد می‌شود؛
- Collection با نام ثابت و نسخه‌دار ساخته می‌شود؛
- Vector size برابر ۳۸۴ و Distance برابر Cosine تنظیم می‌شود؛
- Payload indexهای لازم برای شناسه و فیلترهای اصلی تعریف می‌شوند؛
- اگر Cell دوباره اجرا شود، Collection حذف یا بازسازی نمی‌شود؛
- در اجرای مجدد، تنظیمات Collection موجود Validate می‌شوند.

این مرحله هنوز هیچ Pointی را Upsert نمی‌کند.

</div>

In [6]:
# Code Cell 06 — Create or validate the versioned Qdrant collection

from qdrant_client import models

dense_index_root.mkdir(parents=True, exist_ok=True)

existing_client = globals().get("qdrant_client")
reused_existing_client = False

if existing_client is not None:
    try:
        existing_location = Path(
            str(existing_client._client.location)
        ).resolve()

        assert existing_location == qdrant_storage_path.resolve(), (
            "The existing Qdrant client points to a different storage path: "
            f"{existing_location}"
        )

        existing_client.get_collections()
        qdrant_client = existing_client
        reused_existing_client = True

    except AssertionError:
        raise

    except Exception:
        try:
            existing_client.close()
        except Exception:
            pass

        qdrant_client = QdrantClient(
            path=str(qdrant_storage_path)
        )

else:
    qdrant_client = QdrantClient(
        path=str(qdrant_storage_path)
    )

collection_existed_before = qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)

if not collection_existed_before:
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=release_embedding["dimension"],
            distance=models.Distance.COSINE,
        ),
        on_disk_payload=True,
    )

assert qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)

collection_info = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

vector_config = collection_info.config.params.vectors

assert isinstance(vector_config, models.VectorParams), (
    "Named-vector configuration was not expected for NB14."
)
assert vector_config.size == release_embedding["dimension"]
assert vector_config.distance == models.Distance.COSINE

current_point_count = int(collection_info.points_count or 0)

assert current_point_count <= len(chunk_records), (
    f"Collection contains too many points: {current_point_count}"
)

payload_index_contract = {
    "chunk_id": models.PayloadSchemaType.KEYWORD,
    "source_id": models.PayloadSchemaType.KEYWORD,
    "source_type": models.PayloadSchemaType.KEYWORD,
    "strategy_id": models.PayloadSchemaType.KEYWORD,
    "strategy_version": models.PayloadSchemaType.KEYWORD,
    "role": models.PayloadSchemaType.KEYWORD,
    "chunk_index": models.PayloadSchemaType.INTEGER,
    "dense_index_release_id": models.PayloadSchemaType.KEYWORD,
    "embedding_revision": models.PayloadSchemaType.KEYWORD,
}

payload_index_results = {}

for field_name, field_schema in payload_index_contract.items():
    try:
        operation = qdrant_client.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field_name,
            field_schema=field_schema,
            wait=True,
        )

        payload_index_results[field_name] = {
            "requested_schema": str(field_schema),
            "operation_status": str(
                getattr(operation, "status", "ACKNOWLEDGED")
            ),
        }

    except Exception as exc:
        refreshed_info = qdrant_client.get_collection(
            collection_name=COLLECTION_NAME
        )
        existing_schema = refreshed_info.payload_schema or {}

        assert field_name in existing_schema, (
            f"Payload index creation failed for {field_name}: "
            f"{type(exc).__name__}: {exc}"
        )

        payload_index_results[field_name] = {
            "requested_schema": str(field_schema),
            "operation_status": "ALREADY_EXISTS_AND_VALIDATED",
        }

validated_collection_info = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

validated_vector_config = (
    validated_collection_info.config.params.vectors
)

assert validated_vector_config.size == 384
assert validated_vector_config.distance == models.Distance.COSINE

collection_setup_summary = {
    "storage_path": str(qdrant_storage_path.relative_to(repo_root)),
    "collection_name": COLLECTION_NAME,
    "qdrant_client_reused": reused_existing_client,
    "collection_existed_before": collection_existed_before,
    "vector_size": validated_vector_config.size,
    "distance": str(validated_vector_config.distance),
    "points_before_upsert": int(
        validated_collection_info.points_count or 0
    ),
    "payload_indexes": payload_index_results,
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "collection_setup_status": "PASS",
}

print(json.dumps(collection_setup_summary, ensure_ascii=False, indent=2))

{
  "storage_path": "data/phase2/index/dense_index_v1/qdrant",
  "collection_name": "rag_finance_phase2_dense_v1",
  "qdrant_client_reused": false,
  "collection_existed_before": true,
  "vector_size": 384,
  "distance": "Cosine",
  "points_before_upsert": 3488,
  "payload_indexes": {
    "chunk_id": {
      "requested_schema": "keyword",
      "operation_status": "completed"
    },
    "source_id": {
      "requested_schema": "keyword",
      "operation_status": "completed"
    },
    "source_type": {
      "requested_schema": "keyword",
      "operation_status": "completed"
    },
    "strategy_id": {
      "requested_schema": "keyword",
      "operation_status": "completed"
    },
    "strategy_version": {
      "requested_schema": "keyword",
      "operation_status": "completed"
    },
    "role": {
      "requested_schema": "keyword",
      "operation_status": "completed"
    },
    "chunk_index": {
      "requested_schema": "integer",
      "operation_status": "completed"
    },


/var/folders/c6/kh1v5smn18l3tbcsjpvvqv1h0000gn/T/ipykernel_17832/564699965.py:95: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  operation = qdrant_client.create_payload_index(


<div dir="rtl" align="right">

## مرحله ۶ — Batch Upsert قابل Resume

در این مرحله ۳۴۸۸ Chunk با بردارهای فریز‌شده وارد Collection می‌شوند.

ویژگی‌های اجرا:

- Point ID به‌صورت قطعی از `chunk_id` ساخته می‌شود؛
- تمام Metadataهای اصلی Chunk در Payload حفظ می‌شوند؛
- lineage مربوط به Golden Dataset، Chunk release و Embedding release به هر Point افزوده می‌شود؛
- Upsert در Batchهای ۱۲۸تایی انجام می‌شود؛
- پس از هر Batch یک Checkpoint اتمیک ثبت می‌شود؛
- اجرای مجدد از آخرین Batch تأییدشده ادامه پیدا می‌کند؛
- Upsert با Point ID ثابت idempotent است؛
- هیچ Embedding جدیدی محاسبه نمی‌شود و هیچ API خارجی فراخوانی نمی‌گردد.

در پایان، تعداد Pointهای Collection باید دقیقاً ۳۴۸۸ باشد.

</div>

In [7]:
# Code Cell 07 — Idempotent batch upsert with atomic resume checkpoint

from datetime import datetime, timezone

UPSERT_BATCH_SIZE = dense_index_contract["upsert_policy"]["batch_size"]


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    temporary_path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def build_point_payload(record: dict[str, Any]) -> dict[str, Any]:
    payload = {
        field: record.get(field)
        for field in (
            "chunk_id",
            "chunk_index",
            "source_id",
            "source_type",
            "strategy_id",
            "strategy_version",
            "chunk_contract_version",
            "role",
            "parent_chunk_id",
            "char_count",
            "token_count",
            "content",
            "content_origin",
            "member_document_ids",
            "member_locators",
            "locator_start",
            "locator_end",
            "attribution_context",
            "rights",
            "lineage",
        )
    }

    payload.update(
        {
            "golden_dataset_sha256": handoff[
                "golden_dataset_sha256"
            ],
            "chunk_release_sha256": actual_chunk_sha,
            "embedding_model_id": release_embedding["model_id"],
            "embedding_revision": release_embedding["revision"],
            "embedding_dimension": release_embedding["dimension"],
            "embedding_release_sha256": release_sha,
            "dense_index_release_id": DENSE_INDEX_RELEASE_ID,
        }
    )

    return payload


checkpoint_identity = {
    "schema_version": "1.0",
    "index_release_id": DENSE_INDEX_RELEASE_ID,
    "collection_name": COLLECTION_NAME,
    "handoff_sha256": handoff_sha,
    "embedding_release_sha256": release_sha,
    "chunk_release_sha256": actual_chunk_sha,
    "cache_inventory_sha256": cache_contract["inventory_sha256"],
    "expected_point_count": len(chunk_records),
    "batch_size": UPSERT_BATCH_SIZE,
}

if checkpoint_path.is_file():
    checkpoint = load_json(checkpoint_path)

    for field, expected_value in checkpoint_identity.items():
        assert checkpoint.get(field) == expected_value, (
            f"Checkpoint identity mismatch for {field!r}: "
            f"{checkpoint.get(field)!r} != {expected_value!r}"
        )

    completed_records = int(
        checkpoint.get("completed_records", 0)
    )

else:
    completed_records = 0

    checkpoint = {
        **checkpoint_identity,
        "completed_records": 0,
        "completed_batches": 0,
        "status": "NOT_STARTED",
        "created_at_utc": utc_now_iso(),
        "updated_at_utc": utc_now_iso(),
    }

    atomic_write_json(checkpoint_path, checkpoint)

assert 0 <= completed_records <= len(chunk_records)

# Validate that every point claimed by the checkpoint exists.
if completed_records:
    completed_point_ids = point_ids[:completed_records]
    retrieved_completed_ids = set()

    for batch_start in range(
        0,
        completed_records,
        UPSERT_BATCH_SIZE,
    ):
        batch_ids = completed_point_ids[
            batch_start:batch_start + UPSERT_BATCH_SIZE
        ]

        retrieved = qdrant_client.retrieve(
            collection_name=COLLECTION_NAME,
            ids=batch_ids,
            with_payload=False,
            with_vectors=False,
        )

        retrieved_completed_ids.update(
            str(point.id) for point in retrieved
        )

    missing_checkpoint_points = sorted(
        set(completed_point_ids) - retrieved_completed_ids
    )

    assert not missing_checkpoint_points, (
        "Checkpoint claims points that are absent from Qdrant. "
        f"Examples: {missing_checkpoint_points[:5]}"
    )

started_from_record = completed_records
upserted_this_run = 0

for batch_start in range(
    completed_records,
    len(chunk_records),
    UPSERT_BATCH_SIZE,
):
    batch_end = min(
        batch_start + UPSERT_BATCH_SIZE,
        len(chunk_records),
    )

    points = [
        models.PointStruct(
            id=point_ids[index],
            vector=selected_vectors[index].tolist(),
            payload=build_point_payload(chunk_records[index]),
        )
        for index in range(batch_start, batch_end)
    ]

    operation = qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True,
    )

    operation_status = str(
        getattr(operation, "status", "")
    ).lower()

    assert operation_status in {
        "completed",
        "acknowledged",
    }, f"Unexpected upsert status: {operation_status}"

    completed_records = batch_end
    upserted_this_run += len(points)

    checkpoint = {
        **checkpoint_identity,
        "completed_records": completed_records,
        "completed_batches": (
            completed_records + UPSERT_BATCH_SIZE - 1
        ) // UPSERT_BATCH_SIZE,
        "status": (
            "COMPLETE"
            if completed_records == len(chunk_records)
            else "IN_PROGRESS"
        ),
        "created_at_utc": checkpoint.get(
            "created_at_utc",
            utc_now_iso(),
        ),
        "updated_at_utc": utc_now_iso(),
    }

    atomic_write_json(checkpoint_path, checkpoint)

    print(
        f"Upserted {batch_start}:{batch_end} | "
        f"completed={completed_records}/{len(chunk_records)}"
    )

final_collection_info = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

final_point_count = int(
    final_collection_info.points_count or 0
)

assert completed_records == len(chunk_records)
assert checkpoint["status"] == "COMPLETE"
assert final_point_count == len(chunk_records), (
    f"Point-count mismatch: {final_point_count} != {len(chunk_records)}"
)

upsert_summary = {
    "collection_name": COLLECTION_NAME,
    "started_from_record": started_from_record,
    "upserted_this_run": upserted_this_run,
    "checkpoint_completed_records": completed_records,
    "expected_point_count": len(chunk_records),
    "actual_point_count": final_point_count,
    "checkpoint_status": checkpoint["status"],
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "batch_upsert_status": "PASS",
}

print(json.dumps(upsert_summary, ensure_ascii=False, indent=2))

{
  "collection_name": "rag_finance_phase2_dense_v1",
  "started_from_record": 3488,
  "upserted_this_run": 0,
  "checkpoint_completed_records": 3488,
  "expected_point_count": 3488,
  "actual_point_count": 3488,
  "checkpoint_status": "COMPLETE",
  "paid_api_calls": 0,
  "embedding_recomputation": false,
  "batch_upsert_status": "PASS"
}


<div dir="rtl" align="right">

## مرحله ۷ — ID Parity و Payload/Vector Integrity

Point count به‌تنهایی صحت Index را ثابت نمی‌کند.

در این مرحله:

- تمام Point IDهای مورد انتظار از Qdrant خوانده می‌شوند؛
- نبود Point مفقود یا اضافه بررسی می‌شود؛
- UUID هر Point دوباره از `chunk_id` محاسبه می‌شود؛
- نمونه‌های قطعی از ابتدا، میانه و انتهای Dataset بررسی می‌شوند؛
- Payload ذخیره‌شده با Chunk و Releaseهای فریز‌شده مقایسه می‌شود؛
- Dimension، finite بودن، norm و تطابق بردار ذخیره‌شده با Cache بررسی می‌شود؛
- Chunkهای دارای متن تکراری نیز باید Point ID مستقل داشته باشند.

این مرحله فقط Index ساخته‌شده را می‌خواند و تغییری در آن ایجاد نمی‌کند.

</div>

In [8]:
# Code Cell 08 — Full ID parity and deterministic payload/vector spot-check

expected_point_id_set = set(point_ids)
actual_point_id_set = set()

scroll_offset = None

while True:
    points_page, next_offset = qdrant_client.scroll(
        collection_name=COLLECTION_NAME,
        limit=256,
        offset=scroll_offset,
        with_payload=False,
        with_vectors=False,
    )

    actual_point_id_set.update(
        str(point.id) for point in points_page
    )

    if next_offset is None:
        break

    scroll_offset = next_offset

missing_point_ids = sorted(
    expected_point_id_set - actual_point_id_set
)
unexpected_point_ids = sorted(
    actual_point_id_set - expected_point_id_set
)

assert not missing_point_ids, (
    f"Missing Qdrant points: {missing_point_ids[:5]}"
)
assert not unexpected_point_ids, (
    f"Unexpected Qdrant points: {unexpected_point_ids[:5]}"
)
assert len(actual_point_id_set) == len(chunk_records)

sample_indexes = sorted(
    {
        0,
        1,
        len(chunk_records) // 4,
        len(chunk_records) // 2,
        (3 * len(chunk_records)) // 4,
        len(chunk_records) - 2,
        len(chunk_records) - 1,
    }
)

sample_point_ids = [
    point_ids[index] for index in sample_indexes
]

retrieved_samples = qdrant_client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=sample_point_ids,
    with_payload=True,
    with_vectors=True,
)

retrieved_by_id = {
    str(point.id): point
    for point in retrieved_samples
}

assert set(retrieved_by_id) == set(sample_point_ids)

spot_check_results = []

for index in sample_indexes:
    record = chunk_records[index]
    expected_id = point_ids[index]
    point = retrieved_by_id[expected_id]

    assert expected_id == str(
        uuid.uuid5(
            POINT_UUID_NAMESPACE,
            record["chunk_id"],
        )
    )

    actual_payload = point.payload or {}
    expected_payload = build_point_payload(record)

    missing_payload_fields = sorted(
        set(expected_payload) - set(actual_payload)
    )

    assert not missing_payload_fields, (
        f"Missing payload fields for {record['chunk_id']}: "
        f"{missing_payload_fields}"
    )

    payload_mismatches = {
        field: {
            "expected": expected_payload[field],
            "actual": actual_payload.get(field),
        }
        for field in expected_payload
        if actual_payload.get(field) != expected_payload[field]
    }

    assert not payload_mismatches, (
        f"Payload mismatch for {record['chunk_id']}: "
        f"{payload_mismatches}"
    )

    stored_vector = np.asarray(
        point.vector,
        dtype=np.float32,
    )
    expected_vector = selected_vectors[index]

    assert stored_vector.shape == (384,)
    assert np.isfinite(stored_vector).all()

    stored_norm = float(np.linalg.norm(stored_vector))
    expected_norm = float(np.linalg.norm(expected_vector))
    cosine_similarity = float(
        np.dot(stored_vector, expected_vector)
        / (stored_norm * expected_norm)
    )
    maximum_absolute_delta = float(
        np.max(np.abs(stored_vector - expected_vector))
    )

    assert abs(stored_norm - 1.0) <= 1e-4
    assert cosine_similarity >= 0.999999
    assert maximum_absolute_delta <= 1e-6

    spot_check_results.append(
        {
            "record_index": index,
            "point_id": expected_id,
            "chunk_id": record["chunk_id"],
            "source_id": record["source_id"],
            "source_type": record["source_type"],
            "vector_dimension": int(stored_vector.shape[0]),
            "vector_norm": stored_norm,
            "cosine_similarity_to_cache": cosine_similarity,
            "maximum_absolute_delta": maximum_absolute_delta,
            "payload_status": "PASS",
            "vector_status": "PASS",
        }
    )

canonical_text_groups: dict[str, list[int]] = defaultdict(list)

for index, canonical_text in enumerate(canonical_chunk_texts):
    canonical_text_groups[canonical_text].append(index)

duplicate_text_groups = [
    indexes
    for indexes in canonical_text_groups.values()
    if len(indexes) > 1
]

duplicate_text_point_ids_are_distinct = all(
    len({point_ids[index] for index in indexes}) == len(indexes)
    for indexes in duplicate_text_groups
)

assert duplicate_text_point_ids_are_distinct

integrity_summary = {
    "expected_point_count": len(expected_point_id_set),
    "actual_point_count": len(actual_point_id_set),
    "missing_point_count": len(missing_point_ids),
    "unexpected_point_count": len(unexpected_point_ids),
    "id_parity": "PASS",
    "spot_check_count": len(spot_check_results),
    "spot_checks": spot_check_results,
    "duplicate_canonical_text_groups": len(
        duplicate_text_groups
    ),
    "duplicate_text_point_ids_are_distinct": (
        duplicate_text_point_ids_are_distinct
    ),
    "payload_vector_integrity_status": "PASS",
}

print(json.dumps(integrity_summary, ensure_ascii=False, indent=2))

{
  "expected_point_count": 3488,
  "actual_point_count": 3488,
  "missing_point_count": 0,
  "unexpected_point_count": 0,
  "id_parity": "PASS",
  "spot_check_count": 7,
  "spot_checks": [
    {
      "record_index": 0,
      "point_id": "6f3493cb-b4bf-50d1-bb50-a8eb212e7fa8",
      "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:3c8950efa923b12d11a56f53",
      "source_id": "website:008E61CC",
      "source_type": "website",
      "vector_dimension": 384,
      "vector_norm": 1.0000001192092896,
      "cosine_similarity_to_cache": 0.9999999999999858,
      "maximum_absolute_delta": 0.0,
      "payload_status": "PASS",
      "vector_status": "PASS"
    },
    {
      "record_index": 1,
      "point_id": "66b26953-9e7d-5593-8f16-c962a4330867",
      "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:451d44e731334b417fb8680e",
      "source_id": "website:008E61CC",
      "source_type": "website",
      "vector_dimension": 384,
      "vector_norm": 1.0,
      "cosine_similarit

<div dir="rtl" align="right">

## مرحله ۸ — Snapshot و Restore Test

Dense Index فقط زمانی قابل پذیرش است که نسخه ساخته‌شده قابلیت بازیابی مستقل داشته باشد.

Qdrant Local از Snapshot API سرور استفاده نمی‌کند؛ بنابراین در این مرحله یک Snapshot فایل‌سیستمی سازگار با Storage محلی ساخته می‌شود:

- Client اصلی ابتدا به‌صورت کنترل‌شده بسته می‌شود؛
- Snapshot قطعی و فشرده از Storage ساخته می‌شود؛
- SHA-256 Snapshot ثبت می‌شود؛
- Snapshot داخل یک مسیر موقت مستقل Restore می‌شود؛
- Collection بازیابی‌شده از نظر تنظیمات، Point count، ID parity و نمونه Payload/Vector بررسی می‌شود؛
- محیط Restore موقت پس از Validation پاک می‌شود؛
- Client اصلی دوباره باز و اعتبارسنجی می‌شود؛
- Evidence مستقل Snapshot/Restore ثبت می‌شود.

هیچ Dataset، Embedding یا Artifact فریز‌شده‌ای تغییر نمی‌کند.

</div>

In [9]:
# Code Cell 09 — Deterministic filesystem snapshot and restore validation

import gzip
import shutil
import tarfile
import tempfile

snapshot_directory = dense_index_root / "snapshots"
snapshot_directory.mkdir(parents=True, exist_ok=True)

snapshot_path = (
    snapshot_directory
    / f"{DENSE_INDEX_RELEASE_ID}_qdrant_snapshot.tar.gz"
)
snapshot_temp_path = snapshot_path.with_suffix(
    snapshot_path.suffix + ".tmp"
)

# Release the exclusive local-storage lock before snapshotting.
qdrant_client.close()
qdrant_client = None

assert qdrant_storage_path.is_dir()

storage_files = sorted(
    path
    for path in qdrant_storage_path.rglob("*")
    if path.is_file() and path.name != ".lock"
)

assert storage_files, "Qdrant storage contains no snapshotable files."

with snapshot_temp_path.open("wb") as raw_output:
    with gzip.GzipFile(
        fileobj=raw_output,
        mode="wb",
        mtime=0,
    ) as gzip_output:
        with tarfile.open(
            fileobj=gzip_output,
            mode="w",
        ) as archive:
            for source_path in storage_files:
                archive_name = (
                    Path("qdrant")
                    / source_path.relative_to(qdrant_storage_path)
                )

                tar_info = archive.gettarinfo(
                    str(source_path),
                    arcname=str(archive_name),
                )

                tar_info.uid = 0
                tar_info.gid = 0
                tar_info.uname = ""
                tar_info.gname = ""
                tar_info.mtime = 0

                with source_path.open("rb") as source_handle:
                    archive.addfile(
                        tar_info,
                        fileobj=source_handle,
                    )

snapshot_temp_path.replace(snapshot_path)

snapshot_sha256 = sha256_file(snapshot_path)
snapshot_size_bytes = snapshot_path.stat().st_size

assert snapshot_size_bytes > 0

restore_validation = {}

with tempfile.TemporaryDirectory(
    prefix="nb14_restore_test_",
    dir=str(dense_index_root),
) as temporary_restore_directory:
    restore_root = Path(temporary_restore_directory)

    with tarfile.open(snapshot_path, mode="r:gz") as archive:
        for member in archive.getmembers():
            resolved_target = (
                restore_root / member.name
            ).resolve()

            assert (
                restore_root.resolve()
                in resolved_target.parents
                or resolved_target == restore_root.resolve()
            ), f"Unsafe archive member: {member.name}"

        archive.extractall(restore_root)

    restored_storage_path = restore_root / "qdrant"

    assert restored_storage_path.is_dir()

    restored_client = QdrantClient(
        path=str(restored_storage_path)
    )

    try:
        assert restored_client.collection_exists(
            collection_name=COLLECTION_NAME
        )

        restored_info = restored_client.get_collection(
            collection_name=COLLECTION_NAME
        )

        restored_vector_config = (
            restored_info.config.params.vectors
        )
        restored_point_count = int(
            restored_info.points_count or 0
        )

        assert restored_vector_config.size == 384
        assert (
            restored_vector_config.distance
            == models.Distance.COSINE
        )
        assert restored_point_count == len(chunk_records)

        restored_actual_ids = set()
        restored_offset = None

        while True:
            restored_page, restored_next_offset = (
                restored_client.scroll(
                    collection_name=COLLECTION_NAME,
                    limit=256,
                    offset=restored_offset,
                    with_payload=False,
                    with_vectors=False,
                )
            )

            restored_actual_ids.update(
                str(point.id)
                for point in restored_page
            )

            if restored_next_offset is None:
                break

            restored_offset = restored_next_offset

        assert restored_actual_ids == expected_point_id_set

        restored_samples = restored_client.retrieve(
            collection_name=COLLECTION_NAME,
            ids=sample_point_ids,
            with_payload=True,
            with_vectors=True,
        )

        assert len(restored_samples) == len(sample_point_ids)

        for point in restored_samples:
            point_id = str(point.id)
            original_index = point_ids.index(point_id)
            original_record = chunk_records[original_index]

            assert (
                point.payload["chunk_id"]
                == original_record["chunk_id"]
            )

            restored_vector = np.asarray(
                point.vector,
                dtype=np.float32,
            )

            assert restored_vector.shape == (384,)
            assert np.isfinite(restored_vector).all()

            expected_vector = selected_vectors[original_index]

            restored_cosine = float(
                np.dot(restored_vector, expected_vector)
                / (
                    np.linalg.norm(restored_vector)
                    * np.linalg.norm(expected_vector)
                )
            )

            assert restored_cosine >= 0.999999

        restore_validation = {
            "collection_exists": True,
            "vector_size": restored_vector_config.size,
            "distance": str(restored_vector_config.distance),
            "point_count": restored_point_count,
            "id_parity": "PASS",
            "sample_count": len(restored_samples),
            "payload_vector_spot_check": "PASS",
        }

    finally:
        restored_client.close()

# Reopen and validate the authoritative local index.
qdrant_client = QdrantClient(
    path=str(qdrant_storage_path)
)

authoritative_info_after_restore = (
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    )
)

assert int(
    authoritative_info_after_restore.points_count or 0
) == len(chunk_records)

snapshot_restore_evidence = {
    "schema_version": "1.0",
    "index_release_id": DENSE_INDEX_RELEASE_ID,
    "collection_name": COLLECTION_NAME,
    "snapshot_type": "QDRANT_LOCAL_FILESYSTEM_ARCHIVE",
    "snapshot_path": str(
        snapshot_path.relative_to(repo_root)
    ),
    "snapshot_sha256": snapshot_sha256,
    "snapshot_size_bytes": snapshot_size_bytes,
    "snapshot_source_file_count": len(storage_files),
    "excluded_runtime_files": [".lock"],
    "restore_mode": "ISOLATED_TEMPORARY_DIRECTORY",
    "restore_validation": restore_validation,
    "authoritative_index_reopened": True,
    "authoritative_point_count": int(
        authoritative_info_after_restore.points_count or 0
    ),
    "temporary_restore_copy_removed": True,
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "status": "PASS",
}

atomic_write_json(
    snapshot_evidence_path,
    snapshot_restore_evidence,
)

print(
    json.dumps(
        snapshot_restore_evidence,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "schema_version": "1.0",
  "index_release_id": "dense_index_v1",
  "collection_name": "rag_finance_phase2_dense_v1",
  "snapshot_type": "QDRANT_LOCAL_FILESYSTEM_ARCHIVE",
  "snapshot_path": "data/phase2/index/dense_index_v1/snapshots/dense_index_v1_qdrant_snapshot.tar.gz",
  "snapshot_sha256": "1b07eb4e2c8f02c30235e179456e01c026316ae985f1346fe96f4ae764e587a9",
  "snapshot_size_bytes": 10311767,
  "snapshot_source_file_count": 2,
  "excluded_runtime_files": [
    ".lock"
  ],
  "restore_mode": "ISOLATED_TEMPORARY_DIRECTORY",
  "restore_validation": {
    "collection_exists": true,
    "vector_size": 384,
    "distance": "Cosine",
    "point_count": 3488,
    "id_parity": "PASS",
    "sample_count": 7,
    "payload_vector_spot_check": "PASS"
  },
  "authoritative_index_reopened": true,
  "authoritative_point_count": 3488,
  "temporary_restore_copy_removed": true,
  "paid_api_calls": 0,
  "embedding_recomputation": false,
  "status": "PASS"
}


<div dir="rtl" align="right">

## مرحله ۹ — ساخت Dense Index Manifest

پس از تکمیل Upsert و Restore test، باید هویت منطقی و فیزیکی Index ثبت شود.

Manifest این مرحله شامل موارد زیر است:

- نسخه و نام Collection؛
- Backend، Distance و Dimension؛
- SHA تمام ورودی‌های فریز‌شده؛
- Point count و Payload schema؛
- توزیع Source typeها؛
- fingerprint منطقی تمام Pointها شامل ID، Payload و Vector؛
- inventory فایل‌های Storage؛
- SHA Checkpoint؛
- SHA Snapshot و Restore evidence؛
- سیاست Point ID و Cache key؛
- وضعیت Validationهای انجام‌شده.

Manifest در این مرحله با وضعیت `BUILT_VALIDATED_PENDING_FINAL_GATE` ثبت می‌شود. Freeze نهایی فقط بعد از Gate پایانی انجام خواهد شد.

</div>

In [10]:
# Code Cell 10 — Build canonical dense-index manifest and fingerprints

source_type_counts = Counter(
    record["source_type"]
    for record in chunk_records
)

logical_index_digest = hashlib.sha256()
logical_point_count = 0

for batch_start in range(
    0,
    len(point_ids),
    UPSERT_BATCH_SIZE,
):
    batch_ids = point_ids[
        batch_start:batch_start + UPSERT_BATCH_SIZE
    ]

    retrieved_points = qdrant_client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=batch_ids,
        with_payload=True,
        with_vectors=True,
    )

    retrieved_batch_by_id = {
        str(point.id): point
        for point in retrieved_points
    }

    assert set(retrieved_batch_by_id) == set(batch_ids)

    for point_id in batch_ids:
        point = retrieved_batch_by_id[point_id]

        payload_bytes = json.dumps(
            point.payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")

        vector_bytes = np.asarray(
            point.vector,
            dtype="<f4",
        ).tobytes(order="C")

        for component in (
            point_id.encode("utf-8"),
            payload_bytes,
            vector_bytes,
        ):
            logical_index_digest.update(
                len(component).to_bytes(
                    8,
                    byteorder="big",
                    signed=False,
                )
            )
            logical_index_digest.update(component)

        logical_point_count += 1

assert logical_point_count == len(chunk_records)

logical_index_sha256 = logical_index_digest.hexdigest()

# Close the client before fingerprinting physical storage.
qdrant_client.close()
qdrant_client = None

storage_inventory = []

for storage_file in sorted(
    path
    for path in qdrant_storage_path.rglob("*")
    if path.is_file() and path.name != ".lock"
):
    storage_inventory.append(
        {
            "path": str(
                storage_file.relative_to(qdrant_storage_path)
            ),
            "size_bytes": storage_file.stat().st_size,
            "sha256": sha256_file(storage_file),
        }
    )

assert storage_inventory

storage_inventory_bytes = json.dumps(
    storage_inventory,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

storage_inventory_sha256 = hashlib.sha256(
    storage_inventory_bytes
).hexdigest()

# Reopen the authoritative index after stable storage hashing.
qdrant_client = QdrantClient(
    path=str(qdrant_storage_path)
)

manifest_collection_info = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

assert int(
    manifest_collection_info.points_count or 0
) == len(chunk_records)

checkpoint_sha256 = sha256_file(checkpoint_path)
snapshot_evidence_sha256 = sha256_file(
    snapshot_evidence_path
)

dense_index_manifest = {
    "artifact_type": "DENSE_INDEX_MANIFEST",
    "schema_version": "1.0",
    "status": "BUILT_VALIDATED_PENDING_FINAL_GATE",
    "index_release_id": DENSE_INDEX_RELEASE_ID,
    "created_at_utc": checkpoint["created_at_utc"],
    "backend": {
        "type": "qdrant_local",
        "qdrant_client_version": importlib.metadata.version(
            "qdrant-client"
        ),
        "storage_path": str(
            qdrant_storage_path.relative_to(repo_root)
        ),
        "collection_name": COLLECTION_NAME,
    },
    "vector_config": {
        "dimension": 384,
        "distance": "COSINE",
        "dtype": "float32",
        "normalization": release_embedding["normalization"],
    },
    "point_identity": {
        "source_field": "chunk_id",
        "algorithm": "UUIDv5",
        "namespace": str(POINT_UUID_NAMESPACE),
    },
    "cache_key_contract": {
        "canonicalization": (
            'return " ".join(str(text).split())'
        ),
        "key": (
            "passage: + "
            "sha256(canonical_exact_text(content).encode('utf-8'))"
        ),
    },
    "inputs": {
        "golden_dataset_sha256": handoff[
            "golden_dataset_sha256"
        ],
        "chunk_path": chunk_contract["path"],
        "chunk_sha256": actual_chunk_sha,
        "chunk_count": len(chunk_records),
        "embedding_release_path": str(
            release_path.relative_to(repo_root)
        ),
        "embedding_release_sha256": release_sha,
        "handoff_path": str(
            handoff_path.relative_to(repo_root)
        ),
        "handoff_sha256": handoff_sha,
        "cache_root": cache_contract["root"],
        "cache_inventory_sha256": cache_contract[
            "inventory_sha256"
        ],
    },
    "embedding": {
        "model_id": release_embedding["model_id"],
        "revision": release_embedding["revision"],
        "dimension": release_embedding["dimension"],
        "query_prefix": release_embedding["query_prefix"],
        "passage_prefix": release_embedding["passage_prefix"],
        "normalization": release_embedding["normalization"],
        "snapshot_content_identity_sha256": (
            release_embedding[
                "snapshot_content_identity_sha256"
            ]
        ),
    },
    "index_content": {
        "point_count": logical_point_count,
        "unique_point_ids": len(expected_point_id_set),
        "unique_chunk_ids": len(set(chunk_ids)),
        "unique_canonical_passages": len(
            set(canonical_chunk_texts)
        ),
        "source_type_counts": dict(
            sorted(source_type_counts.items())
        ),
        "logical_index_sha256": logical_index_sha256,
    },
    "payload_schema": {
        field_name: str(field_schema)
        for field_name, field_schema
        in payload_index_contract.items()
    },
    "storage_inventory": {
        "file_count": len(storage_inventory),
        "inventory_sha256": storage_inventory_sha256,
        "files": storage_inventory,
    },
    "checkpoint": {
        "path": str(
            checkpoint_path.relative_to(repo_root)
        ),
        "sha256": checkpoint_sha256,
        "status": checkpoint["status"],
    },
    "snapshot": {
        "path": snapshot_restore_evidence[
            "snapshot_path"
        ],
        "sha256": snapshot_restore_evidence[
            "snapshot_sha256"
        ],
        "restore_evidence_path": str(
            snapshot_evidence_path.relative_to(repo_root)
        ),
        "restore_evidence_sha256": (
            snapshot_evidence_sha256
        ),
        "restore_status": snapshot_restore_evidence[
            "status"
        ],
    },
    "validation": {
        "point_count": "PASS",
        "id_parity": "PASS",
        "payload_spot_check": "PASS",
        "vector_spot_check": "PASS",
        "snapshot_restore": "PASS",
        "paid_api_calls": 0,
        "embedding_recomputation": False,
    },
}

atomic_write_json(
    index_manifest_path,
    dense_index_manifest,
)

manifest_sha256 = sha256_file(index_manifest_path)

manifest_summary = {
    "manifest_path": str(
        index_manifest_path.relative_to(repo_root)
    ),
    "manifest_sha256": manifest_sha256,
    "status": dense_index_manifest["status"],
    "point_count": logical_point_count,
    "logical_index_sha256": logical_index_sha256,
    "storage_inventory_sha256": storage_inventory_sha256,
    "source_type_counts": dict(
        sorted(source_type_counts.items())
    ),
    "checkpoint_sha256": checkpoint_sha256,
    "snapshot_sha256": snapshot_restore_evidence[
        "snapshot_sha256"
    ],
    "restore_evidence_sha256": (
        snapshot_evidence_sha256
    ),
    "manifest_status": "PASS",
}

print(json.dumps(manifest_summary, ensure_ascii=False, indent=2))

{
  "manifest_path": "data/phase2/index/dense_index_v1/dense_index_manifest.json",
  "manifest_sha256": "fe682cfa0df69da2a04f7da1a6545a969abcdd925f614aa295a3f7e2154e9c4b",
  "status": "BUILT_VALIDATED_PENDING_FINAL_GATE",
  "point_count": 3488,
  "logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "storage_inventory_sha256": "c0ff83e70501f4a8bc92fb1cfdf48636d68bce75a3abc4ceb55454c361cd3c6c",
  "source_type_counts": {
    "book": 351,
    "transcript": 2041,
    "website": 1096
  },
  "checkpoint_sha256": "50d49e36516ef30e7edab63344f773968eb28b36438eb893fab1e55f96c6dfc7",
  "snapshot_sha256": "1b07eb4e2c8f02c30235e179456e01c026316ae985f1346fe96f4ae764e587a9",
  "restore_evidence_sha256": "802c0ce3f1306220bed278364d3c67364288f39b20f46fa0553d3710d3c3ff07",
  "manifest_status": "PASS"
}


<div dir="rtl" align="right">

## مرحله ۱۰ — Dense Retrieval Smoke Test

پیش از Freeze نهایی باید ثابت شود Collection فقط قابل ذخیره و Restore نیست، بلکه Dense Search واقعی نیز روی آن درست کار می‌کند.

این آزمون بدون Golden Benchmark جدید و بدون Embedding مجدد انجام می‌شود:

- چند بردار موجود و معتبر به‌عنوان Query probe استفاده می‌شوند؛
- نتایج باید Score متناهی و ترتیب نزولی داشته باشند؛
- Point متناظر یا یکی از Pointهای دارای متن canonical یکسان باید در نتایج برتر حضور داشته باشد؛
- Payloadهای ضروری نتایج بررسی می‌شوند؛
- Metadata filter مستقل برای هر Source type آزموده می‌شود؛
- latency محلی p50 و p95 صرفاً به‌عنوان smoke evidence ثبت می‌شود.

این آزمون جای Benchmarkهای NB16 را نمی‌گیرد و فقط سلامت عملیاتی Dense Index را اثبات می‌کند.

</div>

In [11]:
# Code Cell 11 — Dense search and metadata-filter smoke validation

import time

source_to_indexes: dict[str, list[int]] = defaultdict(list)

for index, record in enumerate(chunk_records):
    source_to_indexes[record["source_type"]].append(index)

probe_indexes = sorted(
    {
        indexes[position]
        for indexes in source_to_indexes.values()
        for position in {
            0,
            len(indexes) // 2,
            len(indexes) - 1,
        }
    }
)

probe_results = []
latencies_ms = []

for probe_index in probe_indexes:
    query_vector = selected_vectors[probe_index]
    canonical_text = canonical_chunk_texts[probe_index]

    equivalent_point_ids = {
        point_ids[index]
        for index in canonical_text_groups[canonical_text]
    }

    started_at = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=10,
        with_payload=True,
        with_vectors=False,
    )

    elapsed_ms = (
        time.perf_counter() - started_at
    ) * 1000

    latencies_ms.append(elapsed_ms)

    result_points = response.points

    assert result_points, (
        f"No dense results for probe index {probe_index}"
    )

    result_ids = [
        str(point.id) for point in result_points
    ]
    result_scores = [
        float(point.score) for point in result_points
    ]

    assert all(
        np.isfinite(result_scores)
    )
    assert all(
        result_scores[index] >= result_scores[index + 1]
        for index in range(len(result_scores) - 1)
    )
    assert equivalent_point_ids.intersection(result_ids), (
        f"Self/equivalent vector not retrieved for {probe_index}"
    )

    assert all(
        isinstance(point.payload.get("chunk_id"), str)
        and point.payload.get("source_id")
        and point.payload.get("source_type")
        and point.payload.get("dense_index_release_id")
        == DENSE_INDEX_RELEASE_ID
        for point in result_points
    )

    first_equivalent_rank = next(
        rank
        for rank, point_id in enumerate(
            result_ids,
            start=1,
        )
        if point_id in equivalent_point_ids
    )

    probe_results.append(
        {
            "probe_index": probe_index,
            "source_type": chunk_records[
                probe_index
            ]["source_type"],
            "chunk_id": chunk_ids[probe_index],
            "equivalent_point_count": len(
                equivalent_point_ids
            ),
            "first_equivalent_rank": (
                first_equivalent_rank
            ),
            "top_score": result_scores[0],
            "result_count": len(result_points),
            "latency_ms": elapsed_ms,
            "status": "PASS",
        }
    )

filter_results = {}

for source_type, indexes in sorted(
    source_to_indexes.items()
):
    probe_index = indexes[len(indexes) // 2]

    filtered_response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=selected_vectors[probe_index].tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="source_type",
                    match=models.MatchValue(
                        value=source_type
                    ),
                )
            ]
        ),
        limit=10,
        with_payload=True,
        with_vectors=False,
    )

    assert filtered_response.points, (
        f"No filtered results for {source_type}"
    )

    returned_source_types = {
        point.payload.get("source_type")
        for point in filtered_response.points
    }

    assert returned_source_types == {source_type}

    filter_results[source_type] = {
        "result_count": len(
            filtered_response.points
        ),
        "returned_source_types": sorted(
            returned_source_types
        ),
        "status": "PASS",
    }

latency_array = np.asarray(
    latencies_ms,
    dtype=float,
)

dense_search_smoke = {
    "probe_count": len(probe_results),
    "source_types_tested": sorted(
        source_to_indexes
    ),
    "all_probes_passed": all(
        result["status"] == "PASS"
        for result in probe_results
    ),
    "probe_results": probe_results,
    "metadata_filter_results": filter_results,
    "observed_local_latency_ms": {
        "minimum": float(
            latency_array.min()
        ),
        "median_p50": float(
            np.percentile(latency_array, 50)
        ),
        "p95": float(
            np.percentile(latency_array, 95)
        ),
        "maximum": float(
            latency_array.max()
        ),
        "measurement_scope": (
            "LOCAL_SMOKE_ONLY_NOT_PRODUCTION_SLO"
        ),
    },
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "dense_search_smoke_status": "PASS",
}

print(
    json.dumps(
        dense_search_smoke,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "probe_count": 9,
  "source_types_tested": [
    "book",
    "transcript",
    "website"
  ],
  "all_probes_passed": true,
  "probe_results": [
    {
      "probe_index": 0,
      "source_type": "website",
      "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:3c8950efa923b12d11a56f53",
      "equivalent_point_count": 1,
      "first_equivalent_rank": 1,
      "top_score": 1.0000000000000004,
      "result_count": 10,
      "latency_ms": 16.992822998872725,
      "status": "PASS"
    },
    {
      "probe_index": 548,
      "source_type": "website",
      "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:323329096d1d31c88e748706",
      "equivalent_point_count": 1,
      "first_equivalent_rank": 1,
      "top_score": 1.0,
      "result_count": 10,
      "latency_ms": 12.056040000970825,
      "status": "PASS"
    },
    {
      "probe_index": 1095,
      "source_type": "website",
      "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:8bfe198b9f0c9eb6908cdfaf",
    

<div dir="rtl" align="right">

## مرحله ۱۱ — Dense Index Gate

تمام Deliverableهای فنی NB14 اکنون اجرا شده‌اند. در این مرحله Evidenceهای مستقل ذخیره و Gate رسمی Dense Index ارزیابی می‌شود.

Gate موارد زیر را بررسی می‌کند:

- صحت قرارداد NB13 → NB14؛
- پوشش کامل Chunk → Cache؛
- Collection config؛
- Point count و ID parity؛
- Payload و Vector integrity؛
- Metadata filtering؛
- Dense search smoke test؛
- Snapshot و Restore؛
- وجود Checkpoint و Manifest؛
- عدم اجرای API پولی؛
- عدم محاسبه مجدد Embedding.

پس از عبور از این Gate، Index در وضعیت Release Candidate قرار می‌گیرد. Freeze نهایی پس از بررسی سلامت خود Notebook انجام می‌شود.

</div>

In [12]:
# Code Cell 12 — Persist evidence and evaluate the formal Dense Index Gate

integrity_evidence_path = (
    dense_index_root / "index_integrity_evidence.json"
)
search_smoke_evidence_path = (
    dense_index_root / "dense_search_smoke_evidence.json"
)

atomic_write_json(
    integrity_evidence_path,
    integrity_summary,
)
atomic_write_json(
    search_smoke_evidence_path,
    dense_search_smoke,
)

integrity_evidence_sha256 = sha256_file(
    integrity_evidence_path
)
search_smoke_evidence_sha256 = sha256_file(
    search_smoke_evidence_path
)

gate_collection_info = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

gate_vector_config = (
    gate_collection_info.config.params.vectors
)
gate_point_count = int(
    gate_collection_info.points_count or 0
)

dense_index_gates = {
    "embedding_release_frozen": (
        release["status"] == "FROZEN_FOR_NB14"
    ),
    "nb14_handoff_ready": (
        handoff["status"] == "READY_FOR_NB14"
    ),
    "golden_sha_bound": (
        release["golden_dataset"]["sha256"]
        == handoff["golden_dataset_sha256"]
    ),
    "chunk_sha_bound": (
        actual_chunk_sha
        == chunk_contract["sha256"]
    ),
    "chunk_count_bound": (
        len(chunk_records)
        == chunk_contract["record_count"]
        == 3488
    ),
    "cache_complete": (
        len(npz_files) == 90
        and len(cache_vectors) == 11433
    ),
    "chunk_cache_mapping_complete": (
        len(missing_chunk_indexes) == 0
    ),
    "collection_exists": (
        qdrant_client.collection_exists(
            collection_name=COLLECTION_NAME
        )
    ),
    "vector_dimension_valid": (
        gate_vector_config.size == 384
    ),
    "distance_valid": (
        gate_vector_config.distance
        == models.Distance.COSINE
    ),
    "point_count_valid": (
        gate_point_count == 3488
    ),
    "id_parity_pass": (
        integrity_summary["id_parity"] == "PASS"
    ),
    "payload_vector_integrity_pass": (
        integrity_summary[
            "payload_vector_integrity_status"
        ] == "PASS"
    ),
    "dense_search_smoke_pass": (
        dense_search_smoke[
            "dense_search_smoke_status"
        ] == "PASS"
    ),
    "metadata_filters_pass": all(
        result["status"] == "PASS"
        for result in dense_search_smoke[
            "metadata_filter_results"
        ].values()
    ),
    "snapshot_restore_pass": (
        snapshot_restore_evidence["status"] == "PASS"
    ),
    "checkpoint_complete": (
        checkpoint["status"] == "COMPLETE"
        and checkpoint["completed_records"] == 3488
    ),
    "manifest_exists": index_manifest_path.is_file(),
    "paid_api_calls_zero": (
        dense_search_smoke["paid_api_calls"] == 0
        and snapshot_restore_evidence[
            "paid_api_calls"
        ] == 0
    ),
    "embedding_recomputation_false": (
        dense_search_smoke[
            "embedding_recomputation"
        ] is False
        and snapshot_restore_evidence[
            "embedding_recomputation"
        ] is False
    ),
}

failed_dense_index_gates = [
    gate_name
    for gate_name, passed in dense_index_gates.items()
    if not passed
]

assert not failed_dense_index_gates, (
    f"Dense Index Gate failed: {failed_dense_index_gates}"
)

dense_index_manifest["status"] = (
    "DENSE_INDEX_GATE_PASS_PENDING_NOTEBOOK_HEALTH"
)

dense_index_manifest["evidence"] = {
    "integrity": {
        "path": str(
            integrity_evidence_path.relative_to(repo_root)
        ),
        "sha256": integrity_evidence_sha256,
    },
    "dense_search_smoke": {
        "path": str(
            search_smoke_evidence_path.relative_to(repo_root)
        ),
        "sha256": search_smoke_evidence_sha256,
    },
    "snapshot_restore": {
        "path": str(
            snapshot_evidence_path.relative_to(repo_root)
        ),
        "sha256": snapshot_evidence_sha256,
    },
}

dense_index_manifest["dense_index_gate"] = {
    "status": "PASS",
    "gates": dense_index_gates,
    "failed_gates": failed_dense_index_gates,
}

atomic_write_json(
    index_manifest_path,
    dense_index_manifest,
)

gate_manifest_sha256 = sha256_file(
    index_manifest_path
)

dense_index_gate_summary = {
    "gate_count": len(dense_index_gates),
    "passed_gate_count": sum(
        dense_index_gates.values()
    ),
    "failed_gates": failed_dense_index_gates,
    "point_count": gate_point_count,
    "logical_index_sha256": logical_index_sha256,
    "integrity_evidence_sha256": (
        integrity_evidence_sha256
    ),
    "search_smoke_evidence_sha256": (
        search_smoke_evidence_sha256
    ),
    "snapshot_restore_evidence_sha256": (
        snapshot_evidence_sha256
    ),
    "manifest_sha256": gate_manifest_sha256,
    "manifest_status": dense_index_manifest["status"],
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "dense_index_gate_status": "PASS",
}

print(
    json.dumps(
        dense_index_gate_summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "gate_count": 20,
  "passed_gate_count": 20,
  "failed_gates": [],
  "point_count": 3488,
  "logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "integrity_evidence_sha256": "f3f066b41c4e63d0ebb48bc5799f32cc18b29949537724ecb6a43bbd13210d22",
  "search_smoke_evidence_sha256": "ce9f8656cd2638fac1a2d9c3d71ccb8a02c6bfd353ee39f9cc67e3a14877f983",
  "snapshot_restore_evidence_sha256": "802c0ce3f1306220bed278364d3c67364288f39b20f46fa0553d3710d3c3ff07",
  "manifest_sha256": "aed2b7f6d96f01ea14d1e58549e475603a09f553b61b57ba638a65707032900b",
  "manifest_status": "DENSE_INDEX_GATE_PASS_PENDING_NOTEBOOK_HEALTH",
  "paid_api_calls": 0,
  "embedding_recomputation": false,
  "dense_index_gate_status": "PASS"
}


<div dir="rtl" align="right">

## مرحله ۱۲ — آزمون بازسازی کامل Dense Index

Dense Index اصلی پیش‌تر تمام کنترل‌های ساخت، یکپارچگی، جست‌وجو، Snapshot و Restore را با موفقیت گذرانده است. بااین‌حال، نقشه عملیاتی کلان Phase 2 در بسته `P2-09` علاوه بر این موارد، قابلیت بازسازی Index از Artifactهای فریز‌شده را نیز الزامی می‌داند.

در این مرحله یک Collection موقت و کاملاً مجزا ساخته می‌شود تا بدون تغییر Collection اصلی، بازسازی Index از ابتدا آزمایش شود.

کنترل‌های این مرحله عبارت‌اند از:

- ثبت وضعیت و fingerprint Collection اصلی پیش از آزمون؛
- ساخت Collection موقت با Dimension و Distance یکسان؛
- بازسازی Payload indexها؛
- Upsert مجدد تمام ۳۴۸۸ Point با همان شناسه‌ها، Payloadها و Vectorهای معتبر؛
- کنترل Point count و برابری کامل شناسه‌ها؛
- مقایسه Payload و Vector تمام Pointها؛
- محاسبه fingerprint منطقی Collection بازسازی‌شده؛
- اثبات برابری آن با fingerprint رسمی Dense Index.

Collection اصلی در این آزمون فقط خوانده می‌شود و هیچ تغییری در آن ایجاد نخواهد شد.

</div>

In [13]:
# Code Cell 13 — Rebuild the dense index in an isolated temporary collection

LIFECYCLE_TEST_COLLECTION = (
    f"{COLLECTION_NAME}__p2_09_lifecycle_test"
)

VECTOR_RTOL = 1e-6
VECTOR_ATOL = 1e-7

if qdrant_client is None:
    qdrant_client = QdrantClient(
        path=str(qdrant_storage_path)
    )

assert qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)

main_collection_before = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

main_point_count_before = int(
    main_collection_before.points_count or 0
)

assert main_point_count_before == len(chunk_records) == 3488
assert len(point_ids) == len(chunk_records)
assert len(selected_vectors) == len(chunk_records)

# The temporary collection is reserved exclusively
# for the P2-09 lifecycle validation.
if qdrant_client.collection_exists(
    collection_name=LIFECYCLE_TEST_COLLECTION
):
    qdrant_client.delete_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    )

qdrant_client.create_collection(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE,
    ),
)

for field_name, field_schema in (
    payload_index_contract.items()
):
    qdrant_client.create_payload_index(
        collection_name=LIFECYCLE_TEST_COLLECTION,
        field_name=field_name,
        field_schema=field_schema,
        wait=True,
    )

rebuilt_source_payloads = {}
rebuilt_upserted_count = 0

for batch_start in range(
    0,
    len(point_ids),
    UPSERT_BATCH_SIZE,
):
    batch_ids = point_ids[
        batch_start:
        batch_start + UPSERT_BATCH_SIZE
    ]

    source_points = qdrant_client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=batch_ids,
        with_payload=True,
        with_vectors=False,
    )

    source_by_id = {
        str(point.id): point
        for point in source_points
    }

    assert set(source_by_id) == set(batch_ids)

    rebuilt_points = []

    for local_offset, point_id in enumerate(batch_ids):
        vector_index = batch_start + local_offset
        source_point = source_by_id[point_id]

        rebuilt_source_payloads[point_id] = (
            source_point.payload
        )

        rebuilt_points.append(
            models.PointStruct(
                id=point_id,
                vector=np.asarray(
                    selected_vectors[vector_index],
                    dtype=np.float32,
                ).tolist(),
                payload=source_point.payload,
            )
        )

    qdrant_client.upsert(
        collection_name=LIFECYCLE_TEST_COLLECTION,
        points=rebuilt_points,
        wait=True,
    )

    rebuilt_upserted_count += len(rebuilt_points)

assert rebuilt_upserted_count == len(point_ids)

rebuilt_collection_info = qdrant_client.get_collection(
    collection_name=LIFECYCLE_TEST_COLLECTION
)

rebuilt_vector_config = (
    rebuilt_collection_info.config.params.vectors
)

rebuilt_point_count = int(
    rebuilt_collection_info.points_count or 0
)

assert rebuilt_vector_config.size == 384
assert (
    rebuilt_vector_config.distance
    == models.Distance.COSINE
)
assert rebuilt_point_count == len(point_ids) == 3488

rebuilt_logical_digest = hashlib.sha256()
rebuilt_id_set = set()
rebuilt_verified_count = 0

rebuilt_payload_mismatches = []
rebuilt_vector_mismatches = []

maximum_absolute_vector_delta = 0.0
minimum_vector_cosine_similarity = 1.0

for batch_start in range(
    0,
    len(point_ids),
    UPSERT_BATCH_SIZE,
):
    batch_ids = point_ids[
        batch_start:
        batch_start + UPSERT_BATCH_SIZE
    ]

    rebuilt_points = qdrant_client.retrieve(
        collection_name=LIFECYCLE_TEST_COLLECTION,
        ids=batch_ids,
        with_payload=True,
        with_vectors=True,
    )

    rebuilt_by_id = {
        str(point.id): point
        for point in rebuilt_points
    }

    assert set(rebuilt_by_id) == set(batch_ids)

    for local_offset, point_id in enumerate(batch_ids):
        vector_index = batch_start + local_offset
        rebuilt_point = rebuilt_by_id[point_id]

        rebuilt_id_set.add(point_id)

        expected_payload = rebuilt_source_payloads[
            point_id
        ]

        expected_vector = np.asarray(
            selected_vectors[vector_index],
            dtype=np.float32,
        )

        actual_vector = np.asarray(
            rebuilt_point.vector,
            dtype=np.float32,
        )

        assert expected_vector.shape == (384,)
        assert actual_vector.shape == (384,)
        assert np.all(np.isfinite(expected_vector))
        assert np.all(np.isfinite(actual_vector))

        if rebuilt_point.payload != expected_payload:
            rebuilt_payload_mismatches.append(
                point_id
            )

        vector_delta = float(
            np.max(
                np.abs(
                    actual_vector - expected_vector
                )
            )
        )

        maximum_absolute_vector_delta = max(
            maximum_absolute_vector_delta,
            vector_delta,
        )

        expected_norm = float(
            np.linalg.norm(expected_vector)
        )
        actual_norm = float(
            np.linalg.norm(actual_vector)
        )

        assert expected_norm > 0
        assert actual_norm > 0

        vector_cosine_similarity = float(
            np.dot(
                expected_vector,
                actual_vector,
            )
            / (
                expected_norm
                * actual_norm
            )
        )

        minimum_vector_cosine_similarity = min(
            minimum_vector_cosine_similarity,
            vector_cosine_similarity,
        )

        if not np.allclose(
            actual_vector,
            expected_vector,
            rtol=VECTOR_RTOL,
            atol=VECTOR_ATOL,
        ):
            rebuilt_vector_mismatches.append(
                {
                    "point_id": point_id,
                    "maximum_absolute_delta": (
                        vector_delta
                    ),
                    "cosine_similarity": (
                        vector_cosine_similarity
                    ),
                }
            )

        payload_bytes = json.dumps(
            rebuilt_point.payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")

        vector_bytes = np.asarray(
            rebuilt_point.vector,
            dtype="<f4",
        ).tobytes(order="C")

        for component in (
            point_id.encode("utf-8"),
            payload_bytes,
            vector_bytes,
        ):
            rebuilt_logical_digest.update(
                len(component).to_bytes(
                    8,
                    byteorder="big",
                    signed=False,
                )
            )
            rebuilt_logical_digest.update(
                component
            )

        rebuilt_verified_count += 1

rebuilt_storage_vector_sha256 = (
    rebuilt_logical_digest.hexdigest()
)

assert rebuilt_verified_count == 3488
assert rebuilt_id_set == expected_point_id_set
assert not rebuilt_payload_mismatches
assert not rebuilt_vector_mismatches

main_collection_after_rebuild = (
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    )
)

main_point_count_after_rebuild = int(
    main_collection_after_rebuild.points_count
    or 0
)

assert (
    main_point_count_after_rebuild
    == main_point_count_before
    == 3488
)

rebuild_evidence = {
    "test_type": (
        "ISOLATED_FULL_DENSE_INDEX_REBUILD"
    ),
    "source_collection": COLLECTION_NAME,
    "test_collection": (
        LIFECYCLE_TEST_COLLECTION
    ),
    "source_point_count_before": (
        main_point_count_before
    ),
    "source_point_count_after": (
        main_point_count_after_rebuild
    ),
    "rebuilt_point_count": (
        rebuilt_point_count
    ),
    "expected_point_count": len(point_ids),
    "rebuilt_upserted_count": (
        rebuilt_upserted_count
    ),
    "rebuilt_verified_count": (
        rebuilt_verified_count
    ),
    "vector_config": {
        "dimension": (
            rebuilt_vector_config.size
        ),
        "distance": str(
            rebuilt_vector_config.distance
        ),
        "status": "PASS",
    },
    "id_parity": "PASS",
    "payload_parity": "PASS",
    "vector_parity": {
        "status": "PASS",
        "method": "numpy.allclose",
        "rtol": VECTOR_RTOL,
        "atol": VECTOR_ATOL,
        "maximum_absolute_delta": (
            maximum_absolute_vector_delta
        ),
        "minimum_cosine_similarity": (
            minimum_vector_cosine_similarity
        ),
        "reason_for_tolerance": (
            "Qdrant COSINE normalization may "
            "introduce non-semantic float32 "
            "differences during re-upsert."
        ),
    },
    "authoritative_logical_index_sha256": (
        logical_index_sha256
    ),
    "rebuilt_storage_vector_sha256": (
        rebuilt_storage_vector_sha256
    ),
    "byte_hash_equality_required": False,
    "source_collection_unchanged": True,
    "temporary_collection_retained_for_next_tests": (
        True
    ),
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "rebuild_status": "PASS",
}

print(
    json.dumps(
        rebuild_evidence,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "test_type": "ISOLATED_FULL_DENSE_INDEX_REBUILD",
  "source_collection": "rag_finance_phase2_dense_v1",
  "test_collection": "rag_finance_phase2_dense_v1__p2_09_lifecycle_test",
  "source_point_count_before": 3488,
  "source_point_count_after": 3488,
  "rebuilt_point_count": 3488,
  "expected_point_count": 3488,
  "rebuilt_upserted_count": 3488,
  "rebuilt_verified_count": 3488,
  "vector_config": {
    "dimension": 384,
    "distance": "Cosine",
    "status": "PASS"
  },
  "id_parity": "PASS",
  "payload_parity": "PASS",
  "vector_parity": {
    "status": "PASS",
    "method": "numpy.allclose",
    "rtol": 1e-06,
    "atol": 1e-07,
    "maximum_absolute_delta": 1.4901161193847656e-08,
    "minimum_cosine_similarity": 0.9999998807907247,
    "reason_for_tolerance": "Qdrant COSINE normalization may introduce non-semantic float32 differences during re-upsert."
  },
  "authoritative_logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "rebuilt_s

<div dir="rtl" align="right">

## مرحله ۱۳ — آزمون چرخه Add، Update و Delete

پس از اثبات بازسازی کامل Dense Index، چرخه تغییر Point روی Collection موقت آزمایش می‌شود.

این آزمون شامل موارد زیر است:

- افزودن یک Point آزمایشی با ID پایدار، Payload معتبر و Vector معتبر؛
- افزایش دقیق Point count پس از Add؛
- بازیابی Point و کنترل Payload و Vector؛
- Update همان Point با Payload و Vector جدید؛
- اثبات ثابت‌ماندن Point count هنگام Update؛
- اثبات جایگزینی صحیح Payload و Vector؛
- حذف Point آزمایشی؛
- بازگشت Point count به مقدار اولیه؛
- اثبات حذف کامل Point؛
- کنترل مجدد Collection اصلی برای اطمینان از عدم تغییر آن.

تمام عملیات تغییردهنده فقط روی Collection موقت انجام می‌شوند. Collection اصلی Dense Index در تمام این مرحله فقط خوانده می‌شود.

</div>

In [14]:
# Code Cell 14 — Validate isolated Add, Update and Delete lifecycle

assert qdrant_client is not None
assert qdrant_client.collection_exists(
    collection_name=LIFECYCLE_TEST_COLLECTION
)
assert qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)

lifecycle_collection_before = (
    qdrant_client.get_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    )
)

lifecycle_base_count = int(
    lifecycle_collection_before.points_count or 0
)

assert lifecycle_base_count == 3488

main_count_before_mutation_tests = int(
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    ).points_count or 0
)

assert main_count_before_mutation_tests == 3488

lifecycle_test_point_id = str(
    uuid.uuid5(
        POINT_UUID_NAMESPACE,
        (
            f"{DENSE_INDEX_RELEASE_ID}:"
            "p2_09:add_update_delete_test"
        ),
    )
)

assert lifecycle_test_point_id not in expected_point_id_set

template_point = qdrant_client.retrieve(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    ids=[point_ids[0]],
    with_payload=True,
    with_vectors=False,
)

assert len(template_point) == 1

add_payload = dict(template_point[0].payload)

add_payload.update(
    {
        "chunk_id": (
            "lifecycle-test:"
            "p2-09:add-update-delete"
        ),
        "lifecycle_test_marker": "ADDED",
        "lifecycle_test_version": 1,
        "dense_index_release_id": (
            DENSE_INDEX_RELEASE_ID
        ),
    }
)

add_vector = np.asarray(
    selected_vectors[0],
    dtype=np.float32,
)

assert add_vector.shape == (384,)
assert np.all(np.isfinite(add_vector))

# ADD
qdrant_client.upsert(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    points=[
        models.PointStruct(
            id=lifecycle_test_point_id,
            vector=add_vector.tolist(),
            payload=add_payload,
        )
    ],
    wait=True,
)

count_after_add = int(
    qdrant_client.get_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    ).points_count or 0
)

assert count_after_add == lifecycle_base_count + 1

added_points = qdrant_client.retrieve(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    ids=[lifecycle_test_point_id],
    with_payload=True,
    with_vectors=True,
)

assert len(added_points) == 1

added_point = added_points[0]
added_vector = np.asarray(
    added_point.vector,
    dtype=np.float32,
)

assert str(added_point.id) == lifecycle_test_point_id
assert added_point.payload == add_payload
assert np.allclose(
    added_vector,
    add_vector,
    rtol=VECTOR_RTOL,
    atol=VECTOR_ATOL,
)

add_maximum_absolute_delta = float(
    np.max(
        np.abs(
            added_vector - add_vector
        )
    )
)

# UPDATE
update_payload = dict(add_payload)

update_payload.update(
    {
        "lifecycle_test_marker": "UPDATED",
        "lifecycle_test_version": 2,
    }
)

update_vector = np.asarray(
    selected_vectors[1],
    dtype=np.float32,
)

assert update_vector.shape == (384,)
assert np.all(np.isfinite(update_vector))
assert not np.allclose(
    add_vector,
    update_vector,
    rtol=VECTOR_RTOL,
    atol=VECTOR_ATOL,
)

qdrant_client.upsert(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    points=[
        models.PointStruct(
            id=lifecycle_test_point_id,
            vector=update_vector.tolist(),
            payload=update_payload,
        )
    ],
    wait=True,
)

count_after_update = int(
    qdrant_client.get_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    ).points_count or 0
)

assert count_after_update == count_after_add

updated_points = qdrant_client.retrieve(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    ids=[lifecycle_test_point_id],
    with_payload=True,
    with_vectors=True,
)

assert len(updated_points) == 1

updated_point = updated_points[0]
updated_vector = np.asarray(
    updated_point.vector,
    dtype=np.float32,
)

assert str(updated_point.id) == lifecycle_test_point_id
assert updated_point.payload == update_payload
assert (
    updated_point.payload["lifecycle_test_marker"]
    == "UPDATED"
)
assert (
    updated_point.payload["lifecycle_test_version"]
    == 2
)
assert np.allclose(
    updated_vector,
    update_vector,
    rtol=VECTOR_RTOL,
    atol=VECTOR_ATOL,
)
assert not np.allclose(
    updated_vector,
    add_vector,
    rtol=VECTOR_RTOL,
    atol=VECTOR_ATOL,
)

update_maximum_absolute_delta = float(
    np.max(
        np.abs(
            updated_vector - update_vector
        )
    )
)

# DELETE
qdrant_client.delete(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    points_selector=models.PointIdsList(
        points=[lifecycle_test_point_id]
    ),
    wait=True,
)

count_after_delete = int(
    qdrant_client.get_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    ).points_count or 0
)

assert count_after_delete == lifecycle_base_count

deleted_point_lookup = qdrant_client.retrieve(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    ids=[lifecycle_test_point_id],
    with_payload=True,
    with_vectors=True,
)

assert deleted_point_lookup == []

# Confirm the canonical collection was never changed.
main_count_after_mutation_tests = int(
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    ).points_count or 0
)

assert (
    main_count_after_mutation_tests
    == main_count_before_mutation_tests
    == 3488
)

# Confirm the temporary collection returned to its
# canonical rebuilt point set after deleting the test point.
remaining_test_points = qdrant_client.retrieve(
    collection_name=LIFECYCLE_TEST_COLLECTION,
    ids=point_ids,
    with_payload=False,
    with_vectors=False,
)

remaining_test_id_set = {
    str(point.id)
    for point in remaining_test_points
}

assert remaining_test_id_set == expected_point_id_set

lifecycle_mutation_evidence = {
    "test_type": "ISOLATED_ADD_UPDATE_DELETE",
    "source_collection": COLLECTION_NAME,
    "test_collection": LIFECYCLE_TEST_COLLECTION,
    "test_point_id": lifecycle_test_point_id,
    "base_point_count": lifecycle_base_count,
    "add": {
        "expected_point_count": (
            lifecycle_base_count + 1
        ),
        "actual_point_count": count_after_add,
        "payload_verified": True,
        "vector_verified": True,
        "maximum_absolute_vector_delta": (
            add_maximum_absolute_delta
        ),
        "status": "PASS",
    },
    "update": {
        "expected_point_count": (
            lifecycle_base_count + 1
        ),
        "actual_point_count": count_after_update,
        "payload_replaced": True,
        "vector_replaced": True,
        "maximum_absolute_vector_delta": (
            update_maximum_absolute_delta
        ),
        "status": "PASS",
    },
    "delete": {
        "expected_point_count": (
            lifecycle_base_count
        ),
        "actual_point_count": count_after_delete,
        "point_absent_after_delete": True,
        "status": "PASS",
    },
    "restored_test_collection_id_parity": (
        "PASS"
    ),
    "source_point_count_before": (
        main_count_before_mutation_tests
    ),
    "source_point_count_after": (
        main_count_after_mutation_tests
    ),
    "source_collection_unchanged": True,
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "lifecycle_mutation_status": "PASS",
}

print(
    json.dumps(
        lifecycle_mutation_evidence,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "test_type": "ISOLATED_ADD_UPDATE_DELETE",
  "source_collection": "rag_finance_phase2_dense_v1",
  "test_collection": "rag_finance_phase2_dense_v1__p2_09_lifecycle_test",
  "test_point_id": "535050f0-9cfd-598d-8b7d-bb4cad7bf987",
  "base_point_count": 3488,
  "add": {
    "expected_point_count": 3489,
    "actual_point_count": 3489,
    "payload_verified": true,
    "vector_verified": true,
    "maximum_absolute_vector_delta": 1.4901161193847656e-08,
    "status": "PASS"
  },
  "update": {
    "expected_point_count": 3489,
    "actual_point_count": 3489,
    "payload_replaced": true,
    "vector_replaced": true,
    "maximum_absolute_vector_delta": 7.450580596923828e-09,
    "status": "PASS"
  },
  "delete": {
    "expected_point_count": 3488,
    "actual_point_count": 3488,
    "point_absent_after_delete": true,
    "status": "PASS"
  },
  "restored_test_collection_id_parity": "PASS",
  "source_point_count_before": 3488,
  "source_point_count_after": 3488,
  "source_collection_unc

<div dir="rtl" align="right">

## مرحله ۱۴ — آزمون Re-index و Idempotency

در این مرحله تمام Pointهای Canonical بار دیگر روی Collection موقت موجود Upsert می‌شوند. هدف این است که ثابت شود اجرای مجدد Indexing:

- Point تکراری ایجاد نمی‌کند؛
- تعداد Pointها را تغییر نمی‌دهد؛
- شناسه‌ها را حفظ می‌کند؛
- Payloadها را تغییر نمی‌دهد؛
- Vectorها را در محدوده عددی مجاز حفظ می‌کند؛
- Collection اصلی را تحت تأثیر قرار نمی‌دهد.

این آزمون، رفتار Re-index روی Collection موجود و خاصیت Idempotency فرایند Indexing را بررسی می‌کند. اختلاف بایتی ناچیز ناشی از Normalization مجدد `COSINE` به‌عنوان تغییر معنایی تلقی نمی‌شود و Vectorها با تلورانس عددی صریح ارزیابی می‌شوند.

</div>

In [15]:
# Code Cell 15 — Validate full re-index and idempotent upsert behavior

assert qdrant_client is not None
assert qdrant_client.collection_exists(
    collection_name=LIFECYCLE_TEST_COLLECTION
)
assert qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)

reindex_count_before = int(
    qdrant_client.get_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    ).points_count or 0
)

main_count_before_reindex = int(
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    ).points_count or 0
)

assert reindex_count_before == 3488
assert main_count_before_reindex == 3488

pre_reindex_payloads = {}
pre_reindex_vectors = {}

for batch_start in range(
    0,
    len(point_ids),
    UPSERT_BATCH_SIZE,
):
    batch_ids = point_ids[
        batch_start:
        batch_start + UPSERT_BATCH_SIZE
    ]

    existing_points = qdrant_client.retrieve(
        collection_name=LIFECYCLE_TEST_COLLECTION,
        ids=batch_ids,
        with_payload=True,
        with_vectors=True,
    )

    existing_by_id = {
        str(point.id): point
        for point in existing_points
    }

    assert set(existing_by_id) == set(batch_ids)

    for point_id in batch_ids:
        existing_point = existing_by_id[point_id]

        pre_reindex_payloads[point_id] = (
            existing_point.payload
        )
        pre_reindex_vectors[point_id] = np.asarray(
            existing_point.vector,
            dtype=np.float32,
        )

assert len(pre_reindex_payloads) == 3488
assert len(pre_reindex_vectors) == 3488

reindexed_upsert_count = 0

for batch_start in range(
    0,
    len(point_ids),
    UPSERT_BATCH_SIZE,
):
    batch_ids = point_ids[
        batch_start:
        batch_start + UPSERT_BATCH_SIZE
    ]

    reindex_points = []

    for local_offset, point_id in enumerate(batch_ids):
        vector_index = batch_start + local_offset

        reindex_points.append(
            models.PointStruct(
                id=point_id,
                vector=np.asarray(
                    selected_vectors[vector_index],
                    dtype=np.float32,
                ).tolist(),
                payload=rebuilt_source_payloads[
                    point_id
                ],
            )
        )

    qdrant_client.upsert(
        collection_name=LIFECYCLE_TEST_COLLECTION,
        points=reindex_points,
        wait=True,
    )

    reindexed_upsert_count += len(reindex_points)

assert reindexed_upsert_count == 3488

reindex_count_after = int(
    qdrant_client.get_collection(
        collection_name=LIFECYCLE_TEST_COLLECTION
    ).points_count or 0
)

assert reindex_count_after == reindex_count_before == 3488

post_reindex_id_set = set()
post_reindex_verified_count = 0
post_reindex_payload_mismatches = []
post_reindex_vector_mismatches = []

maximum_delta_from_pre_reindex = 0.0
maximum_delta_from_canonical_vector = 0.0
minimum_post_reindex_cosine_similarity = 1.0

post_reindex_digest = hashlib.sha256()

for batch_start in range(
    0,
    len(point_ids),
    UPSERT_BATCH_SIZE,
):
    batch_ids = point_ids[
        batch_start:
        batch_start + UPSERT_BATCH_SIZE
    ]

    reindexed_points = qdrant_client.retrieve(
        collection_name=LIFECYCLE_TEST_COLLECTION,
        ids=batch_ids,
        with_payload=True,
        with_vectors=True,
    )

    reindexed_by_id = {
        str(point.id): point
        for point in reindexed_points
    }

    assert set(reindexed_by_id) == set(batch_ids)

    for local_offset, point_id in enumerate(batch_ids):
        vector_index = batch_start + local_offset
        reindexed_point = reindexed_by_id[point_id]

        post_reindex_id_set.add(point_id)

        actual_payload = reindexed_point.payload
        actual_vector = np.asarray(
            reindexed_point.vector,
            dtype=np.float32,
        )

        expected_payload = rebuilt_source_payloads[
            point_id
        ]
        pre_reindex_vector = pre_reindex_vectors[
            point_id
        ]
        canonical_vector = np.asarray(
            selected_vectors[vector_index],
            dtype=np.float32,
        )

        assert actual_vector.shape == (384,)
        assert np.all(np.isfinite(actual_vector))

        if actual_payload != expected_payload:
            post_reindex_payload_mismatches.append(
                point_id
            )

        delta_from_pre_reindex = float(
            np.max(
                np.abs(
                    actual_vector
                    - pre_reindex_vector
                )
            )
        )

        delta_from_canonical_vector = float(
            np.max(
                np.abs(
                    actual_vector
                    - canonical_vector
                )
            )
        )

        maximum_delta_from_pre_reindex = max(
            maximum_delta_from_pre_reindex,
            delta_from_pre_reindex,
        )

        maximum_delta_from_canonical_vector = max(
            maximum_delta_from_canonical_vector,
            delta_from_canonical_vector,
        )

        canonical_norm = float(
            np.linalg.norm(canonical_vector)
        )
        actual_norm = float(
            np.linalg.norm(actual_vector)
        )

        assert canonical_norm > 0
        assert actual_norm > 0

        cosine_similarity = float(
            np.dot(
                canonical_vector,
                actual_vector,
            )
            / (
                canonical_norm
                * actual_norm
            )
        )

        minimum_post_reindex_cosine_similarity = min(
            minimum_post_reindex_cosine_similarity,
            cosine_similarity,
        )

        if not np.allclose(
            actual_vector,
            canonical_vector,
            rtol=VECTOR_RTOL,
            atol=VECTOR_ATOL,
        ):
            post_reindex_vector_mismatches.append(
                {
                    "point_id": point_id,
                    "maximum_absolute_delta": (
                        delta_from_canonical_vector
                    ),
                    "cosine_similarity": (
                        cosine_similarity
                    ),
                }
            )

        payload_bytes = json.dumps(
            actual_payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")

        vector_bytes = np.asarray(
            actual_vector,
            dtype="<f4",
        ).tobytes(order="C")

        for component in (
            point_id.encode("utf-8"),
            payload_bytes,
            vector_bytes,
        ):
            post_reindex_digest.update(
                len(component).to_bytes(
                    8,
                    byteorder="big",
                    signed=False,
                )
            )
            post_reindex_digest.update(
                component
            )

        post_reindex_verified_count += 1

post_reindex_storage_sha256 = (
    post_reindex_digest.hexdigest()
)

assert post_reindex_verified_count == 3488
assert post_reindex_id_set == expected_point_id_set
assert not post_reindex_payload_mismatches
assert not post_reindex_vector_mismatches

main_count_after_reindex = int(
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    ).points_count or 0
)

assert (
    main_count_after_reindex
    == main_count_before_reindex
    == 3488
)

reindex_evidence = {
    "test_type": (
        "FULL_REINDEX_AND_IDEMPOTENCY"
    ),
    "source_collection": COLLECTION_NAME,
    "test_collection": (
        LIFECYCLE_TEST_COLLECTION
    ),
    "point_count_before": reindex_count_before,
    "upserted_record_count": (
        reindexed_upsert_count
    ),
    "point_count_after": reindex_count_after,
    "duplicate_points_created": False,
    "id_parity": "PASS",
    "payload_parity": "PASS",
    "vector_parity": {
        "status": "PASS",
        "method": "numpy.allclose",
        "rtol": VECTOR_RTOL,
        "atol": VECTOR_ATOL,
        "maximum_delta_from_pre_reindex": (
            maximum_delta_from_pre_reindex
        ),
        "maximum_delta_from_canonical_vector": (
            maximum_delta_from_canonical_vector
        ),
        "minimum_cosine_similarity": (
            minimum_post_reindex_cosine_similarity
        ),
    },
    "verified_point_count": (
        post_reindex_verified_count
    ),
    "post_reindex_storage_sha256": (
        post_reindex_storage_sha256
    ),
    "source_point_count_before": (
        main_count_before_reindex
    ),
    "source_point_count_after": (
        main_count_after_reindex
    ),
    "source_collection_unchanged": True,
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "reindex_idempotency_status": "PASS",
}

print(
    json.dumps(
        reindex_evidence,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "test_type": "FULL_REINDEX_AND_IDEMPOTENCY",
  "source_collection": "rag_finance_phase2_dense_v1",
  "test_collection": "rag_finance_phase2_dense_v1__p2_09_lifecycle_test",
  "point_count_before": 3488,
  "upserted_record_count": 3488,
  "point_count_after": 3488,
  "duplicate_points_created": false,
  "id_parity": "PASS",
  "payload_parity": "PASS",
  "vector_parity": {
    "status": "PASS",
    "method": "numpy.allclose",
    "rtol": 1e-06,
    "atol": 1e-07,
    "maximum_delta_from_pre_reindex": 1.4901161193847656e-08,
    "maximum_delta_from_canonical_vector": 1.4901161193847656e-08,
    "minimum_cosine_similarity": 0.9999998807907247
  },
  "verified_point_count": 3488,
  "post_reindex_storage_sha256": "0803fbbe9f6f4e5323c5f583714660ad7cf1c873cc7d734c04d0713b6cde0ccd",
  "source_point_count_before": 3488,
  "source_point_count_after": 3488,
  "source_collection_unchanged": true,
  "paid_api_calls": 0,
  "embedding_recomputation": false,
  "reindex_idempotency_status": "PASS"
}

<div dir="rtl" align="right">

## مرحله ۱۵ — Gate تکمیلی چرخه عمر Index در P2-09

آزمون‌های تکمیلی نقشه عملیاتی `P2-09` اکنون اجرا شده‌اند. این Gate بررسی می‌کند که:

- بازسازی کامل Index از ابتدا موفق بوده است؛
- تمام ۳۴۸۸ شناسه و Payload بدون اختلاف بازسازی شده‌اند؛
- Vectorها با تلورانس عددی صریح معادل بوده‌اند؛
- عملیات Add موفق بوده است؛
- عملیات Update بدون ایجاد Point تکراری موفق بوده است؛
- عملیات Delete و بازگشت Point count موفق بوده است؛
- Re-index کامل خاصیت Idempotency داشته است؛
- Collection اصلی در تمام آزمون‌ها بدون تغییر باقی مانده است؛
- هیچ API پولی یا محاسبه مجدد Embedding انجام نشده است.

پس از ذخیره Evidence، Collection موقت حذف می‌شود. سپس وجود و Point count Collection اصلی دوباره کنترل خواهد شد.

عبور از این Gate نشان می‌دهد الزامات چرخه عمر Index در نقشه عملیاتی کلان، علاوه بر الزامات نقشه ماژولار NB14، پوشش داده شده‌اند. Freeze نهایی پس از بررسی سلامت کامل Notebook انجام می‌شود.

</div>

In [16]:
# Code Cell 16 — Persist P2-09 lifecycle evidence and evaluate its gate

lifecycle_evidence_path = (
    dense_index_root
    / "index_lifecycle_evidence.json"
)

lifecycle_evidence = {
    "artifact_type": (
        "DENSE_INDEX_LIFECYCLE_EVIDENCE"
    ),
    "schema_version": "1.0",
    "index_release_id": DENSE_INDEX_RELEASE_ID,
    "source_collection": COLLECTION_NAME,
    "temporary_test_collection": (
        LIFECYCLE_TEST_COLLECTION
    ),
    "rebuild": rebuild_evidence,
    "add_update_delete": (
        lifecycle_mutation_evidence
    ),
    "reindex_idempotency": reindex_evidence,
    "vector_comparison_policy": {
        "method": "numpy.allclose",
        "rtol": VECTOR_RTOL,
        "atol": VECTOR_ATOL,
        "byte_hash_equality_required": False,
        "reason": (
            "Qdrant COSINE normalization may "
            "introduce non-semantic float32 "
            "differences during repeated upsert."
        ),
    },
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "status": "PASS",
}

atomic_write_json(
    lifecycle_evidence_path,
    lifecycle_evidence,
)

lifecycle_evidence_sha256 = sha256_file(
    lifecycle_evidence_path
)

p2_09_lifecycle_gates = {
    "isolated_rebuild_pass": (
        rebuild_evidence["rebuild_status"]
        == "PASS"
    ),
    "rebuild_point_count_valid": (
        rebuild_evidence["rebuilt_point_count"]
        == 3488
    ),
    "rebuild_id_parity_pass": (
        rebuild_evidence["id_parity"]
        == "PASS"
    ),
    "rebuild_payload_parity_pass": (
        rebuild_evidence["payload_parity"]
        == "PASS"
    ),
    "rebuild_vector_parity_pass": (
        rebuild_evidence["vector_parity"][
            "status"
        ]
        == "PASS"
    ),
    "add_pass": (
        lifecycle_mutation_evidence["add"][
            "status"
        ]
        == "PASS"
    ),
    "update_pass": (
        lifecycle_mutation_evidence["update"][
            "status"
        ]
        == "PASS"
    ),
    "delete_pass": (
        lifecycle_mutation_evidence["delete"][
            "status"
        ]
        == "PASS"
    ),
    "reindex_idempotency_pass": (
        reindex_evidence[
            "reindex_idempotency_status"
        ]
        == "PASS"
    ),
    "no_duplicate_points_created": (
        reindex_evidence[
            "duplicate_points_created"
        ]
        is False
    ),
    "source_collection_unchanged": (
        rebuild_evidence[
            "source_collection_unchanged"
        ]
        is True
        and lifecycle_mutation_evidence[
            "source_collection_unchanged"
        ]
        is True
        and reindex_evidence[
            "source_collection_unchanged"
        ]
        is True
    ),
    "paid_api_calls_zero": (
        lifecycle_evidence["paid_api_calls"]
        == 0
    ),
    "embedding_recomputation_false": (
        lifecycle_evidence[
            "embedding_recomputation"
        ]
        is False
    ),
}

failed_p2_09_lifecycle_gates = [
    gate_name
    for gate_name, passed
    in p2_09_lifecycle_gates.items()
    if not passed
]

assert not failed_p2_09_lifecycle_gates, (
    "P2-09 lifecycle gate failed: "
    f"{failed_p2_09_lifecycle_gates}"
)

assert qdrant_client.collection_exists(
    collection_name=LIFECYCLE_TEST_COLLECTION
)

qdrant_client.delete_collection(
    collection_name=LIFECYCLE_TEST_COLLECTION
)

assert not qdrant_client.collection_exists(
    collection_name=LIFECYCLE_TEST_COLLECTION
)

assert qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)

canonical_collection_after_cleanup = (
    qdrant_client.get_collection(
        collection_name=COLLECTION_NAME
    )
)

canonical_point_count_after_cleanup = int(
    canonical_collection_after_cleanup.points_count
    or 0
)

canonical_vector_config_after_cleanup = (
    canonical_collection_after_cleanup
    .config.params.vectors
)

assert canonical_point_count_after_cleanup == 3488
assert (
    canonical_vector_config_after_cleanup.size
    == 384
)
assert (
    canonical_vector_config_after_cleanup.distance
    == models.Distance.COSINE
)

# Load the persisted manifest to avoid depending on
# stale in-memory status from an earlier finalization run.
dense_index_manifest = json.loads(
    index_manifest_path.read_text(
        encoding="utf-8"
    )
)

dense_index_manifest["status"] = (
    "P2_09_LIFECYCLE_GATE_PASS_"
    "PENDING_NOTEBOOK_HEALTH"
)

dense_index_manifest.pop(
    "final_status",
    None,
)
dense_index_manifest.pop(
    "execution_health",
    None,
)

dense_index_manifest["p2_09_lifecycle"] = {
    "status": "PASS",
    "evidence": {
        "path": str(
            lifecycle_evidence_path.relative_to(
                repo_root
            )
        ),
        "sha256": lifecycle_evidence_sha256,
    },
    "gates": p2_09_lifecycle_gates,
    "failed_gates": (
        failed_p2_09_lifecycle_gates
    ),
    "temporary_collection_deleted": True,
    "canonical_collection_unchanged": True,
}

atomic_write_json(
    index_manifest_path,
    dense_index_manifest,
)

lifecycle_gate_manifest_sha256 = sha256_file(
    index_manifest_path
)

p2_09_lifecycle_gate_summary = {
    "gate_count": len(
        p2_09_lifecycle_gates
    ),
    "passed_gate_count": sum(
        p2_09_lifecycle_gates.values()
    ),
    "failed_gates": (
        failed_p2_09_lifecycle_gates
    ),
    "tests": {
        "rebuild": "PASS",
        "add": "PASS",
        "update": "PASS",
        "delete": "PASS",
        "reindex": "PASS",
        "idempotency": "PASS",
    },
    "canonical_collection": COLLECTION_NAME,
    "canonical_point_count": (
        canonical_point_count_after_cleanup
    ),
    "canonical_collection_unchanged": True,
    "temporary_collection_deleted": True,
    "lifecycle_evidence": {
        "path": str(
            lifecycle_evidence_path.relative_to(
                repo_root
            )
        ),
        "sha256": lifecycle_evidence_sha256,
    },
    "manifest_sha256": (
        lifecycle_gate_manifest_sha256
    ),
    "manifest_status": (
        dense_index_manifest["status"]
    ),
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "p2_09_lifecycle_gate_status": "PASS",
}

print(
    json.dumps(
        p2_09_lifecycle_gate_summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "gate_count": 13,
  "passed_gate_count": 13,
  "failed_gates": [],
  "tests": {
    "rebuild": "PASS",
    "add": "PASS",
    "update": "PASS",
    "delete": "PASS",
    "reindex": "PASS",
    "idempotency": "PASS"
  },
  "canonical_collection": "rag_finance_phase2_dense_v1",
  "canonical_point_count": 3488,
  "canonical_collection_unchanged": true,
  "temporary_collection_deleted": true,
  "lifecycle_evidence": {
    "path": "data/phase2/index/dense_index_v1/index_lifecycle_evidence.json",
    "sha256": "bcea3bda1ee7ad86c9c6ca40ab674aa9bc23394a668dce86a99e5b03a1a3d2ef"
  },
  "manifest_sha256": "98968c0a710a874657633f2dbe30072926071f9193931e61b991fcbea94ca0e0",
  "manifest_status": "P2_09_LIFECYCLE_GATE_PASS_PENDING_NOTEBOOK_HEALTH",
  "paid_api_calls": 0,
  "embedding_recomputation": false,
  "p2_09_lifecycle_gate_status": "PASS"
}


<div dir="rtl" align="right">

# نتیجه نهایی NB14 — Dense Index V1

هدف Notebook 14 ساختن یک Dense Index صرفاً قابل جست‌وجو نبود؛ هدف، تولید یک Index نسخه‌دار، قابل ردیابی، قابل بازسازی و قابل تحویل به مراحل بعدی Retrieval بود.

## نتیجه اجرای نقشه ماژولار NB14

تمام مأموریت‌های رسمی نقشه ماژولار انجام شدند:

1. Chunk Release و Embedding Release معتبر بارگذاری و به SHAهای فریز‌شده متصل شدند.
2. Collection نسخه‌دار Qdrant ساخته شد.
3. Dimension برابر ۳۸۴ و Distance برابر COSINE تأیید شد.
4. Payload schema و Metadata indexها ساخته شدند.
5. تمام ۳۴۸۸ Point با Batch Upsert و Checkpoint وارد Index شدند.
6. Point count و ID parity کامل تأیید شد.
7. Payload و Vector integrity بررسی شد.
8. Snapshot مستقل ساخته و Restore آن آزمایش شد.
9. Manifest، Evidence و fingerprintهای Index ثبت شدند.
10. Dense Index Gate با ۲۰ کنترل از ۲۰ کنترل عبور کرد.

## نتیجه اجرای الزامات تکمیلی P2-09

برای پوشش کامل‌تر نقشه عملیاتی کلان، چرخه عمر Index نیز روی Collection موقت و ایزوله آزمایش شد:

- Rebuild کامل ۳۴۸۸ Point از ابتدا؛
- برابری کامل ID و Payload؛
- معادل‌بودن عددی Vectorها با تلورانس صریح؛
- Add یک Point جدید؛
- Update همان Point بدون ایجاد Duplicate؛
- Delete و بازگشت Point count؛
- Re-index کامل؛
- اثبات Idempotency؛
- حذف Collection موقت؛
- اثبات عدم تغییر Collection اصلی.

اختلاف بسیار کوچک بایتی Vectorها پس از Upsert مجدد، ناشی از Normalization عددی COSINE است. بیشترین اختلاف مشاهده‌شده در حد `1.49e-08` بوده و تمام Vectorها با تلورانس تعریف‌شده و Cosine similarity نزدیک به یک تأیید شده‌اند.

## وضعیت نهایی

- تعداد Pointهای Canonical: ۳۴۸۸
- منابع: Website، Book و Transcript
- Dense Search: PASS
- Metadata filtering: PASS
- Snapshot و Restore: PASS
- Rebuild: PASS
- Add / Update / Delete: PASS
- Re-index و Idempotency: PASS
- API پولی: صفر
- محاسبه مجدد Embedding: انجام نشده
- Collection اصلی: بدون تغییر
- وضعیت نهایی: `PASS_COMPLETE_FROZEN`

بنابراین NB14 هم به هدف رسمی نقشه ماژولار و هم به الزامات مرتبط Index در بسته عملیاتی `P2-09` رسیده است.

این Notebook شامل Sparse Retrieval، Hybrid Fusion یا Reranking نیست؛ این موارد عمداً خارج از Scope آن هستند:

- مرحله بعدی اجرای پروژه: `NB15 — Sparse Index`
- مصرف‌کننده بعدی Dense Index: `NB16 — Hybrid Retrieval`
- Reranking: `NB17`

Dense Index از این مرحله به بعد Frozen است و مراحل بعد نباید آن را تغییر دهند. هر مصرف‌کننده باید Release و Handoff نسخه‌دار آن را Load و Validate کند.

</div>

In [18]:
# Code Cell 17 — Final health, release, acceptance and freeze

import ast
import re

notebook_path = (
    repo_root / "notebooks/phase2/14_dense_index.ipynb"
)

assert notebook_path.is_file(), (
    f"NB14 notebook not found: {notebook_path}"
)

notebook_payload = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

code_cells = [
    cell
    for cell in notebook_payload.get("cells", [])
    if cell.get("cell_type") == "code"
]

code_numbers = []
syntax_errors = []
error_outputs = []
unexecuted_code_cells = []
forbidden_api_cells = []
api_enabled_true_cells = []

for notebook_cell_index, cell in enumerate(code_cells):
    source = "".join(cell.get("source", []))

    number_match = re.search(
        r"#\s*Code Cell\s*(\d+)",
        source,
        flags=re.IGNORECASE,
    )

    code_number = (
        int(number_match.group(1))
        if number_match
        else None
    )

    if code_number is not None:
        code_numbers.append(code_number)

    try:
        ast.parse(source)
    except SyntaxError as exc:
        syntax_errors.append(
            {
                "code_cell": code_number,
                "line": exc.lineno,
                "message": exc.msg,
            }
        )

    if (
        "chat.completions" + ".create" in source
        or "embeddings" + ".create" in source
    ):
        forbidden_api_cells.append(code_number)

    if re.search(
        r"\b[A-Z0-9_]*API[A-Z0-9_]*\s*=\s*True\b",
        source,
    ):
        api_enabled_true_cells.append(
            code_number
        )

    for output in cell.get("outputs", []):
        if (
            output.get("output_type") == "error"
            and code_number != 17
        ):
            error_outputs.append(
                {
                    "code_cell": code_number,
                    "ename": output.get("ename"),
                    "evalue": output.get("evalue"),
                }
            )

    if (
        cell.get("execution_count") is None
        and code_number != 17
    ):
        unexecuted_code_cells.append(code_number)

assert code_numbers == list(range(1, 18)), (
    "Code Cell numbering is not 01→17: "
    f"{code_numbers}"
)
assert len(code_numbers) == len(set(code_numbers))
assert not syntax_errors, syntax_errors
assert not error_outputs, error_outputs
assert not unexecuted_code_cells, (
    unexecuted_code_cells
)
assert not forbidden_api_cells, forbidden_api_cells
assert not api_enabled_true_cells, (
    api_enabled_true_cells
)

assert qdrant_client is not None
assert qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
)
assert not qdrant_client.collection_exists(
    collection_name=LIFECYCLE_TEST_COLLECTION
)

final_collection_info = qdrant_client.get_collection(
    collection_name=COLLECTION_NAME
)

final_vector_config = (
    final_collection_info.config.params.vectors
)

final_point_count = int(
    final_collection_info.points_count or 0
)

assert final_point_count == 3488
assert final_vector_config.size == 384
assert (
    final_vector_config.distance
    == models.Distance.COSINE
)

modular_nb14_deliverables = {
    "canonical_inputs_loaded": (
        release["status"] == "FROZEN_FOR_NB14"
        and handoff["status"] == "READY_FOR_NB14"
    ),
    "versioned_collection_created": (
        qdrant_client.collection_exists(
            collection_name=COLLECTION_NAME
        )
    ),
    "distance_and_dimension_valid": (
        final_vector_config.size == 384
        and final_vector_config.distance
        == models.Distance.COSINE
    ),
    "payload_schema_created": (
        len(payload_index_contract) == 9
    ),
    "batch_upsert_resume_complete": (
        checkpoint["status"] == "COMPLETE"
        and checkpoint["completed_records"] == 3488
    ),
    "point_count_and_id_parity_pass": (
        final_point_count == 3488
        and integrity_summary["id_parity"]
        == "PASS"
    ),
    "payload_vector_integrity_pass": (
        integrity_summary[
            "payload_vector_integrity_status"
        ]
        == "PASS"
    ),
    "snapshot_restore_test_pass": (
        snapshot_restore_evidence["status"]
        == "PASS"
    ),
    "index_manifest_created": (
        index_manifest_path.is_file()
    ),
    "dense_index_gate_pass": (
        dense_index_gate_summary[
            "dense_index_gate_status"
        ]
        == "PASS"
    ),
}

p2_09_operational_deliverables = {
    "full_rebuild_pass": (
        rebuild_evidence["rebuild_status"]
        == "PASS"
    ),
    "add_pass": (
        lifecycle_mutation_evidence["add"][
            "status"
        ]
        == "PASS"
    ),
    "update_pass": (
        lifecycle_mutation_evidence["update"][
            "status"
        ]
        == "PASS"
    ),
    "delete_pass": (
        lifecycle_mutation_evidence["delete"][
            "status"
        ]
        == "PASS"
    ),
    "reindex_pass": (
        reindex_evidence[
            "reindex_idempotency_status"
        ]
        == "PASS"
    ),
    "idempotency_pass": (
        reindex_evidence[
            "duplicate_points_created"
        ]
        is False
    ),
    "temporary_collection_removed": (
        not qdrant_client.collection_exists(
            collection_name=(
                LIFECYCLE_TEST_COLLECTION
            )
        )
    ),
    "canonical_collection_unchanged": (
        final_point_count == 3488
        and rebuild_evidence[
            "source_collection_unchanged"
        ]
        is True
        and lifecycle_mutation_evidence[
            "source_collection_unchanged"
        ]
        is True
        and reindex_evidence[
            "source_collection_unchanged"
        ]
        is True
    ),
    "lifecycle_gate_pass": (
        p2_09_lifecycle_gate_summary[
            "p2_09_lifecycle_gate_status"
        ]
        == "PASS"
    ),
}

failed_modular_deliverables = [
    name
    for name, passed
    in modular_nb14_deliverables.items()
    if not passed
]

failed_operational_deliverables = [
    name
    for name, passed
    in p2_09_operational_deliverables.items()
    if not passed
]

assert not failed_modular_deliverables, (
    failed_modular_deliverables
)
assert not failed_operational_deliverables, (
    failed_operational_deliverables
)

execution_health = {
    "syntax_errors": syntax_errors,
    "error_outputs_excluding_active_final_cell": (
        error_outputs
    ),
    "unexecuted_code_cells_excluding_active_final_cell": (
        unexecuted_code_cells
    ),
    "code_numbers": code_numbers,
    "forbidden_api_cells": forbidden_api_cells,
    "api_enabled_true_cells": (
        api_enabled_true_cells
    ),
    "status": "PASS",
}

# Close Qdrant before calculating the final physical
# storage inventory after lifecycle-test cleanup.
qdrant_client.close()
qdrant_client = None

final_storage_inventory = []

for storage_file in sorted(
    path
    for path in qdrant_storage_path.rglob("*")
    if path.is_file() and path.name != ".lock"
):
    final_storage_inventory.append(
        {
            "path": str(
                storage_file.relative_to(
                    qdrant_storage_path
                )
            ),
            "size_bytes": (
                storage_file.stat().st_size
            ),
            "sha256": sha256_file(
                storage_file
            ),
        }
    )

assert final_storage_inventory

final_storage_inventory_bytes = json.dumps(
    final_storage_inventory,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

final_storage_inventory_sha256 = (
    hashlib.sha256(
        final_storage_inventory_bytes
    ).hexdigest()
)

dense_index_manifest = json.loads(
    index_manifest_path.read_text(
        encoding="utf-8"
    )
)

dense_index_manifest["status"] = (
    "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
)
dense_index_manifest["final_status"] = (
    "PASS_COMPLETE_FROZEN"
)

dense_index_manifest["storage_inventory"] = {
    "file_count": len(
        final_storage_inventory
    ),
    "inventory_sha256": (
        final_storage_inventory_sha256
    ),
    "files": final_storage_inventory,
    "scope": (
        "FINAL_STORAGE_STATE_AFTER_"
        "LIFECYCLE_TEST_CLEANUP"
    ),
}

dense_index_manifest["execution_health"] = (
    execution_health
)

dense_index_manifest[
    "modular_nb14_deliverables"
] = {
    "checks": modular_nb14_deliverables,
    "failed": failed_modular_deliverables,
    "status": "PASS",
}

dense_index_manifest[
    "p2_09_operational_deliverables"
] = {
    "checks": p2_09_operational_deliverables,
    "failed": failed_operational_deliverables,
    "status": "PASS",
}

dense_index_manifest["p2_09_lifecycle"] = {
    "status": "PASS",
    "evidence": {
        "path": str(
            lifecycle_evidence_path.relative_to(
                repo_root
            )
        ),
        "sha256": (
            lifecycle_evidence_sha256
        ),
    },
    "gates": p2_09_lifecycle_gates,
    "failed_gates": (
        failed_p2_09_lifecycle_gates
    ),
    "temporary_collection_deleted": True,
    "canonical_collection_unchanged": True,
}

dense_index_manifest["scope_boundary"] = {
    "included": [
        "dense_index",
        "metadata_filtering",
        "snapshot_restore",
        "rebuild",
        "add",
        "update",
        "delete",
        "reindex",
        "idempotency",
    ],
    "excluded": [
        "sparse_index",
        "hybrid_retrieval",
        "reranking",
    ],
    "next_execution_notebook": (
        "15_sparse_index.ipynb"
    ),
    "dense_index_consumer_notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
}

atomic_write_json(
    index_manifest_path,
    dense_index_manifest,
)

final_manifest_sha256 = sha256_file(
    index_manifest_path
)

canonical_artifact_root = (
    repo_root / "artifacts/phase2/dense_index"
)
canonical_artifact_root.mkdir(
    parents=True,
    exist_ok=True,
)

reports_phase2_root = repo_root / "reports/phase2"
reports_phase2_root.mkdir(
    parents=True,
    exist_ok=True,
)

dense_release_path = (
    canonical_artifact_root
    / "dense_index_release_v1.json"
)

nb16_handoff_path = (
    canonical_artifact_root
    / "nb16_dense_index_handoff_v1.json"
)

acceptance_path = (
    reports_phase2_root
    / "dense_index_acceptance_v1.json"
)

dense_index_release = {
    "artifact_type": "DENSE_INDEX_RELEASE",
    "schema_version": "1.1",
    "release_version": DENSE_INDEX_RELEASE_ID,
    "status": (
        "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
    ),
    "final_status": "PASS_COMPLETE_FROZEN",
    "backend": {
        "type": "qdrant_local",
        "qdrant_client_version": (
            importlib.metadata.version(
                "qdrant-client"
            )
        ),
        "storage_path": str(
            qdrant_storage_path.relative_to(
                repo_root
            )
        ),
        "collection_name": COLLECTION_NAME,
    },
    "vector": {
        "dimension": 384,
        "distance": "COSINE",
        "dtype": "float32",
        "normalization": release_embedding[
            "normalization"
        ],
    },
    "point_count": final_point_count,
    "logical_index_sha256": (
        logical_index_sha256
    ),
    "storage_inventory_sha256": (
        final_storage_inventory_sha256
    ),
    "manifest": {
        "path": str(
            index_manifest_path.relative_to(
                repo_root
            )
        ),
        "sha256": final_manifest_sha256,
    },
    "snapshot": {
        "path": snapshot_restore_evidence[
            "snapshot_path"
        ],
        "sha256": snapshot_restore_evidence[
            "snapshot_sha256"
        ],
        "restore_status": "PASS",
    },
    "lifecycle_validation": {
        "path": str(
            lifecycle_evidence_path.relative_to(
                repo_root
            )
        ),
        "sha256": lifecycle_evidence_sha256,
        "rebuild": "PASS",
        "add": "PASS",
        "update": "PASS",
        "delete": "PASS",
        "reindex": "PASS",
        "idempotency": "PASS",
    },
    "inputs": dense_index_manifest["inputs"],
    "embedding": (
        dense_index_manifest["embedding"]
    ),
    "point_identity": (
        dense_index_manifest["point_identity"]
    ),
    "payload_schema": (
        dense_index_manifest["payload_schema"]
    ),
    "constraints": {
        "paid_api_required": False,
        "embedding_recomputation_required": (
            False
        ),
        "dense_index_mutation_allowed": False,
        "sparse_index_included": False,
        "hybrid_retrieval_included": False,
        "reranking_included": False,
    },
}

atomic_write_json(
    dense_release_path,
    dense_index_release,
)

dense_release_sha256 = sha256_file(
    dense_release_path
)

nb16_dense_handoff = {
    "artifact_type": (
        "NB16_DENSE_INDEX_HANDOFF"
    ),
    "schema_version": "1.1",
    "status": "READY_FOR_NB16",
    "dense_index_release": {
        "path": str(
            dense_release_path.relative_to(
                repo_root
            )
        ),
        "sha256": dense_release_sha256,
    },
    "index_release_id": DENSE_INDEX_RELEASE_ID,
    "collection_name": COLLECTION_NAME,
    "storage_path": str(
        qdrant_storage_path.relative_to(
            repo_root
        )
    ),
    "point_count": final_point_count,
    "logical_index_sha256": (
        logical_index_sha256
    ),
    "dimension": 384,
    "distance": "COSINE",
    "golden_dataset_sha256": handoff[
        "golden_dataset_sha256"
    ],
    "chunk_sha256": actual_chunk_sha,
    "embedding_release_sha256": release_sha,
    "lifecycle_validation_status": "PASS",
    "constraints": {
        "do_not_recompute_embeddings": True,
        "do_not_mutate_dense_index": True,
        "reranking_required": False,
    },
}

atomic_write_json(
    nb16_handoff_path,
    nb16_dense_handoff,
)

nb16_handoff_sha256 = sha256_file(
    nb16_handoff_path
)

dense_index_acceptance = {
    "artifact_type": (
        "NB14_DENSE_INDEX_ACCEPTANCE"
    ),
    "schema_version": "1.1",
    "notebook": (
        "notebooks/phase2/"
        "14_dense_index.ipynb"
    ),
    "objective": {
        "modular_nb14_scope_achieved": True,
        "p2_09_index_lifecycle_achieved": True,
        "objective_status": "PASS",
    },
    "execution_health": execution_health,
    "modular_nb14_deliverables": {
        "checks": modular_nb14_deliverables,
        "failed": failed_modular_deliverables,
        "status": "PASS",
    },
    "p2_09_operational_deliverables": {
        "checks": (
            p2_09_operational_deliverables
        ),
        "failed": (
            failed_operational_deliverables
        ),
        "status": "PASS",
    },
    "release": {
        "path": str(
            dense_release_path.relative_to(
                repo_root
            )
        ),
        "sha256": dense_release_sha256,
        "status": dense_index_release[
            "status"
        ],
    },
    "nb16_handoff": {
        "path": str(
            nb16_handoff_path.relative_to(
                repo_root
            )
        ),
        "sha256": nb16_handoff_sha256,
        "status": nb16_dense_handoff[
            "status"
        ],
    },
    "lifecycle_evidence": {
        "path": str(
            lifecycle_evidence_path.relative_to(
                repo_root
            )
        ),
        "sha256": lifecycle_evidence_sha256,
    },
    "point_count": final_point_count,
    "logical_index_sha256": (
        logical_index_sha256
    ),
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "next_execution_notebook": (
        "15_sparse_index.ipynb"
    ),
    "dense_index_consumer_notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
    "release_status": (
        "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
    ),
    "final_status": "PASS_COMPLETE_FROZEN",
    "status": "PASS",
}

atomic_write_json(
    acceptance_path,
    dense_index_acceptance,
)

acceptance_sha256 = sha256_file(
    acceptance_path
)

final_summary = {
    "nb14_objective": "PASS",
    "modular_nb14_deliverables": "PASS",
    "p2_09_index_lifecycle": "PASS",
    "execution_health": "PASS",
    "dense_index_gate": "PASS",
    "lifecycle_gate": "PASS",
    "point_count": final_point_count,
    "logical_index_sha256": (
        logical_index_sha256
    ),
    "storage_inventory_sha256": (
        final_storage_inventory_sha256
    ),
    "manifest": {
        "path": str(
            index_manifest_path.relative_to(
                repo_root
            )
        ),
        "sha256": final_manifest_sha256,
    },
    "release": {
        "path": str(
            dense_release_path.relative_to(
                repo_root
            )
        ),
        "sha256": dense_release_sha256,
        "status": dense_index_release[
            "status"
        ],
    },
    "nb16_handoff": {
        "path": str(
            nb16_handoff_path.relative_to(
                repo_root
            )
        ),
        "sha256": nb16_handoff_sha256,
        "status": nb16_dense_handoff[
            "status"
        ],
    },
    "acceptance": {
        "path": str(
            acceptance_path.relative_to(
                repo_root
            )
        ),
        "sha256": acceptance_sha256,
    },
    "lifecycle_evidence": {
        "path": str(
            lifecycle_evidence_path.relative_to(
                repo_root
            )
        ),
        "sha256": lifecycle_evidence_sha256,
    },
    "temporary_collection_deleted": True,
    "canonical_collection_unchanged": True,
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "next_execution_notebook": (
        "15_sparse_index.ipynb"
    ),
    "dense_index_consumer_notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
    "final_status": "PASS_COMPLETE_FROZEN",
}

print(
    json.dumps(
        final_summary,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nNB14 FINAL HEALTH: "
    "PASS — COMPLETE / LIFECYCLE-VALIDATED / FROZEN"
)

{
  "nb14_objective": "PASS",
  "modular_nb14_deliverables": "PASS",
  "p2_09_index_lifecycle": "PASS",
  "execution_health": "PASS",
  "dense_index_gate": "PASS",
  "lifecycle_gate": "PASS",
  "point_count": 3488,
  "logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "storage_inventory_sha256": "c0ff83e70501f4a8bc92fb1cfdf48636d68bce75a3abc4ceb55454c361cd3c6c",
  "manifest": {
    "path": "data/phase2/index/dense_index_v1/dense_index_manifest.json",
    "sha256": "739539f7abdfd0df435389e103658e28244b078f425f33878458034058bfae47"
  },
  "release": {
    "path": "artifacts/phase2/dense_index/dense_index_release_v1.json",
    "sha256": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
    "status": "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
  },
  "nb16_handoff": {
    "path": "artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json",
    "sha256": "da7f812a79a21b1876c74668a548badb0f271b2d18ac2f1b2cf9663f789f301f",
    "status": "R